# Pocket Agent
> **Build a fully functional local coding agent from scratch.**
> 12 chapters · one notebook · no API costs.

Each chapter adds exactly one capability.
Run a chapter's cells top-to-bottom and you get a working agent at the end of that chapter.

| Ch | Title | What you gain |
|----|-------|---------------|
| 1 | Hello Ollama | Talk to a local LLM from Python |
| 2 | Context is a Budget | Understand why context management exists |
| 3 | Give it a Map | Load AGENTS.md as an advisory project manifest |
| 4 | Glob + JIT Reads | Navigate a file tree without bulk-loading files |
| 5 | Fuzzy Scoring | Rank retrieved files, not just find them |
| 6 | Grep | Find code by content, not just filename |
| 7 | Microcompaction | Hot tail / cold storage — survive long sessions |
| 8 | Semantic RAG | Retrieval that understands meaning, not just keywords |
| 9 | Full Pipeline | One `run(query, repo)` call does everything |
| 9b | Web UI | Optional browser interface at localhost:8000 |
| 10 | Write + Diff | The agent modifies files on disk |
| 11 | Agent Loop | Autonomous read → plan → write → verify loop |
| 12 | Test Generation | Agent writes and verifies its own tests |

---

## Global Configuration

**Edit this cell before running anything else.**
Every chapter reads these variables, so you only need to set them once.

| Variable | What it controls |
|----------|-----------------|
| `OLLAMA_BASE_URL` | Where your Ollama server is listening |
| `OLLAMA_MODEL` | The chat model to use (must be pulled) |
| `OLLAMA_EMBED` | The embedding model (needed from Chapter 8) |
| `REPO_ROOT` | The local repo the agent will navigate |
| `TOKEN_BUDGET` | Context-window size in tokens |
| `USE_WEB_UI` | Switch to `True` after completing Chapter 9b |

In [176]:
# ── Global configuration — edit before running ───────────────────────────
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "qwen3-coder-next:cloud"  # any model you have pulled
OLLAMA_EMBED    = "nomic-embed-text"  # embedding model, needed from Ch 8
REPO_ROOT       = "."                 # repo the agent will navigate
TOKEN_BUDGET    = 262144               # context-window size in tokens
USE_WEB_UI      = False               # set True after completing Chapter 9b

---
## Chapter 1 — Hello Ollama

**Goal:** establish the two primitives everything else builds on —
a function that talks to Ollama, and a status panel that makes the
agent's internal state legible at every step.

**You will:**
- Verify Ollama is reachable
- Write `chat()` — the single function every later chapter calls
- Write `show_panel()` — a `rich` terminal panel showing the token
  budget, retrieved files, active strategy, and assembled prompt size
- Run a test query end-to-end

### 1.1 Dependencies

Chapter 1 only needs `requests` (HTTP to Ollama) and `rich` (terminal UI).
Later chapters will `pip install` their own additions at the top of their section.

In [118]:
import subprocess, sys

_CH1_DEPS = ["requests", "rich"]

for _pkg in _CH1_DEPS:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", _pkg],
        stdout=subprocess.DEVNULL,
    )

print("Chapter 1 dependencies ready:", _CH1_DEPS)

Chapter 1 dependencies ready: ['requests', 'rich']


### 1.2 Checking Ollama is running

Before sending any messages we probe the `/api/tags` endpoint.
That endpoint returns the list of locally pulled models — a useful
sanity check that both the server and the model we want are present.

In [119]:
import requests

def ping_ollama() -> tuple[bool, list[str] | str]:
    """
    Return (ok, info).
    ok=True  → info is a list of pulled model names.
    ok=False → info is the error message.
    """
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        return True, models
    except Exception as exc:
        return False, str(exc)


ok, info = ping_ollama()

if ok:
    print(f"Ollama is running.")
    print(f"Pulled models: {info}")
    if OLLAMA_MODEL not in " ".join(info):
        print(f"\n  WARNING: '{OLLAMA_MODEL}' not found in pulled models.")
        print(f"  Run: ollama pull {OLLAMA_MODEL}")
else:
    print(f"Cannot reach Ollama at {OLLAMA_BASE_URL}")
    print(f"Error: {info}")
    print("Start it with: ollama serve")

Ollama is running.
Pulled models: ['qwen3-coder-next:cloud', 'nomic-embed-text:latest', 'qwen3.5:4b', 'deepseek-v3.1:671b-cloud', 'qwen3-coder:480b-cloud', 'codellama:7b']


### 1.3 The `chat()` function

`chat()` is the only function that ever calls Ollama.
Every chapter — from the simplest single-turn query to the full
autonomous agent loop — goes through this one function.

It takes a standard OpenAI-style `messages` list and returns
the reply text plus the token count Ollama reports.
The token count is what populates the budget bar in the status panel.

In [120]:
def chat(
    messages: list[dict],
    model: str = OLLAMA_MODEL,
) -> tuple[str, int]:
    """
    Send *messages* to Ollama, return (reply_text, tokens_used).

    tokens_used = prompt_eval_count + eval_count as reported by Ollama.
    This is the number that drives the token-budget bar.
    """
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"num_ctx": TOKEN_BUDGET},
    }
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    data   = r.json()
    reply  = data["message"]["content"]
    tokens = data.get("prompt_eval_count", 0) + data.get("eval_count", 0)
    return reply, tokens

### 1.4 The status panel

The panel is the same in every chapter — only the data it receives changes.
It shows four things:

| Row | What it shows |
|-----|--------------|
| **Token Budget** | A coloured progress bar. Green < 60 %, yellow < 85 %, red above. |
| **Retrieved** | Every file the agent loaded, tagged HOT (in prompt) or COLD (offloaded). |
| **Strategy** | Which retrieval strategy was used: glob, fuzzy, grep, or semantic. |
| **Prompt size** | The assembled prompt in tokens, *before* the LLM call. |

`show_panel()` is intentionally stateless — you call it with the current
snapshot before the LLM call and again after, so you can see both.

In [121]:
from rich.console import Console
from rich.panel   import Panel
from rich.table   import Table
from rich.text    import Text

console = Console()


def _budget_bar(used: int, total: int, width: int = 32) -> Text:
    """Render a coloured progress bar for the token budget."""
    pct    = used / total if total > 0 else 0
    filled = int(width * pct)
    color  = "green" if pct < 0.6 else ("yellow" if pct < 0.85 else "red")

    bar = Text()
    bar.append("█" * filled,          style=color)
    bar.append("░" * (width - filled), style="dim")
    bar.append(f"  {used:,} / {total:,} tokens")
    return bar


def show_panel(
    query:        str,
    token_used:   int,
    token_budget: int        = TOKEN_BUDGET,
    files:        list[dict] | None = None,
    strategy:     str        = "none",
    prompt_size:  int        = 0,
) -> None:
    """
    Print the Pocket Agent status panel to the terminal.

    Parameters
    ----------
    query        : the user's query (shown in the panel title)
    token_used   : tokens consumed so far (from Ollama or estimated)
    token_budget : total context-window size (default TOKEN_BUDGET)
    files        : list of dicts {"path": str, "hot": bool}
                   hot=True  → file is in the active prompt
                   hot=False → file is in cold storage
    strategy     : retrieval strategy name shown in the panel
    prompt_size  : estimated assembled-prompt size in tokens
    """
    files = files or []

    grid = Table.grid(padding=(0, 2))
    grid.add_column(style="bold cyan", no_wrap=True, min_width=14)
    grid.add_column()

    # Token budget row
    grid.add_row("Token Budget", _budget_bar(token_used, token_budget))
    grid.add_row("", "")

    # Retrieved files
    if files:
        lines = Text()
        for f in files:
            badge = (
                Text(" HOT  ", style="bold white on red")
                if f["hot"]
                else Text(" COLD ", style="white on blue")
            )
            lines.append_text(badge)
            lines.append(f"  {f['path']}\n")
        grid.add_row("Retrieved", lines)
    else:
        grid.add_row("Retrieved", Text("(none)", style="dim"))

    grid.add_row("Strategy",    Text(strategy,                  style="bold magenta"))
    grid.add_row("Prompt size", Text(f"{prompt_size:,} tokens", style="dim"))

    console.print(
        Panel(
            grid,
            title=(
                f"[bold]Pocket Agent[/bold]  ·  "
                f"[italic]{query[:70]}{'…' if len(query) > 70 else ''}[/italic]"
            ),
            border_style="bright_blue",
            expand=False,
        )
    )

### 1.5 Try it

Send a single question to Ollama and observe the panel
before the call (prompt assembled, no reply yet) and after
(real token count from Ollama).

The token count before the call is a rough estimate (`len(text) // 4`).
From Chapter 2 onward we'll use a proper tokeniser.

In [122]:
query    = "What is a context window, and why does its size matter?"
messages = [{"role": "user", "content": query}]

# Rough pre-call estimate: 1 token ≈ 4 characters
prompt_size = len(query) // 4

console.rule("[bold]Before LLM call[/bold]")
show_panel(
    query        = query,
    token_used   = prompt_size,
    strategy     = "none",
    prompt_size  = prompt_size,
)

console.print("\n[dim]Sending query to Ollama…[/dim]\n")
reply, tokens_used = chat(messages)

console.rule("[bold]After LLM call[/bold]")
show_panel(
    query       = query,
    token_used  = tokens_used,
    strategy    = "none",
    prompt_size = prompt_size,
)

console.print(f"\n[bold green]Response:[/bold green]\n{reply}")

───────────────────────────────────────────────── Before LLM call ─────────────────────────────────────────────────

╭─ Pocket Agent  ·  What is a context window, and why does its size matter? ─╮
│ Token Budget      ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  13 / 4,096 tokens      │
│                                                                            │
│ Retrieved         (none)                                                   │
│ Strategy          none                                                     │
│ Prompt size       13 tokens                                                │
╰────────────────────────────────────────────────────────────────────────────╯

Sending query to Ollama…

───────────────────────────────────────────────── After LLM call ──────────────────────────────────────────────────

╭─ Pocket Agent  ·  What is a context window, and why does its size matter? ─╮
│ Token Budget      ███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  458 / 4,096 tokens     │
│                                                                            │
│ Retrieved         (none)                                                   │
│ Strategy          none                                                     │
│ Prompt size       13 tokens                                                │
╰────────────────────────────────────────────────────────────────────────────╯

Response:
A **context window** refers to the maximum amount of text (tokens) that a language model can process at once—i.e., 
the total number of tokens (words, subwords, or characters) it can consider as input and output in a single 
interaction.

Think of it as the model’s "working memory" capacity for that interaction. For example:
- A model with a 4K-token context window can handle ~3,000 words of input *plus* output combined.
- A model with a 128K-token context window can handle much longer documents or conversations.

### Why size matters:
1. **Handling long documents**  
   Larger context windows let models process entire books, legal contracts, or codebases in one go—without manual 
summarization or chunking.

2. **Maintaining conversation coherence**  
   In chat, a larger window preserves more of the dialogue history, helping the model remember earlier points, 
avoid contradictions, and maintain consistency over long conversations.

3. **Improved reasoning and retrieval**  
   For tasks like RAG (Retrieval-Augmented Generation), larger windows can hold more retrieved passages, enabling 
better synthesis and more accurate answers.

4. **Reduced data loss**  
   With small windows, long inputs must be truncated or split, risking loss of crucial context (e.g., losing the 
beginning of a story or error in a mid-document reference).

5. **Efficiency vs. capability trade-off**  
   Very large context windows can be computationally expensive (quadratic or worse scaling), so model architects 
balance size, speed, and cost.

💡 **Real-world analogy**:  
Imagine reading a complex manual. If your working memory only holds 2 pages at a time (small context), you’ll 
struggle to connect ideas between page 1 and page 10. With a memory that holds 50 pages (large context), you can 
synthesize the whole manual on the fly.

Many modern models now offer *extended* or *unlimited* context (via techniques like attention skipping or external 
memory), but knowing the *effective* limit for a given task is key for reliable performance.

---
## Chapter 2 — Context is a Budget

**Goal:** understand *why* context management exists before we build any of it.

Every token you load — user message, system prompt, file content, conversation history — occupies space in the context window. When that space runs out, one of two things happens: the model silently truncates early content (it forgets the beginning of the conversation), or the API returns an error.

A coding agent that naively loads every file in a repo will hit the wall on the first non-trivial query.

**You will:**
- Replace the inline `len // 4` guess with a named `count_tokens()` function
- Watch a multi-turn conversation burn through its budget turn by turn
- Measure the token cost of real files on disk
- See the panel turn yellow then red as budget pressure rises
- Understand the three strategies agents use to stay inside the window (foreshadowing Ch 3–8)

### 2.1 `count_tokens()` — one place to estimate token cost

We can't use tiktoken here (that's OpenAI-specific), and loading a full
HuggingFace tokeniser just to count tokens is overkill for a local agent.

The `÷ 4` heuristic (1 token ≈ 4 characters in English prose) is accurate to
within ~15 % for most code and documentation. That's precise enough to drive
a budget bar — we don't need exact counts, we need *early warning*.

Wrapping it in a named function means we can swap the implementation once
(e.g. call Ollama's `/api/tokenize` endpoint) without touching any of the
chapter code that calls it.

In [123]:
def count_tokens(text: str) -> int:
    """
    Estimate token count for *text*.

    Uses the 4-characters-per-token heuristic — accurate to ~15 % for
    English prose and source code.  Good enough to drive a budget bar.

    Swap the body for a real tokeniser call if you need precision:
        # example: use Ollama's tokenise endpoint
        # r = requests.post(f"{OLLAMA_BASE_URL}/api/tokenize",
        #                   json={"model": OLLAMA_MODEL, "content": text})
        # return len(r.json()["tokens"])
    """
    return max(1, len(text) // 4)


# Quick sanity check
_sample = "The quick brown fox jumps over the lazy dog."
print(f"Sample: {len(_sample)} chars → {count_tokens(_sample)} tokens  "
      f"(GPT-4 tokeniser gives ~11 for this sentence)")

Sample: 44 chars → 11 tokens  (GPT-4 tokeniser gives ~11 for this sentence)


### 2.2 Watching a conversation burn its budget

Each turn appends the user message *and* the assistant reply to the
`messages` list before the next call.  Ollama sees the full list every time —
that's how it maintains conversation context, but it also means every reply
you get back is added to your next prompt.

Run this cell and watch the budget bar grow with each exchange.

In [124]:
TURNS = [
    "What is a Python list comprehension? Give a one-line example.",
    "Now show a dictionary comprehension that squares numbers 1–5.",
    "What is the difference between a shallow copy and a deep copy?",
    "Show me a minimal example where a shallow copy causes a bug.",
]

messages: list[dict] = []
tokens_used = 0

for i, question in enumerate(TURNS, start=1):
    messages.append({"role": "user", "content": question})

    # Estimate prompt size before the call
    prompt_text = " ".join(m["content"] for m in messages)
    prompt_size = count_tokens(prompt_text)

    console.rule(f"[bold]Turn {i} — before call[/bold]")
    show_panel(
        query        = question,
        token_used   = tokens_used,
        strategy     = "multi-turn",
        prompt_size  = prompt_size,
    )

    reply, tokens_used = chat(messages)
    messages.append({"role": "assistant", "content": reply})

    console.rule(f"[bold]Turn {i} — after call[/bold]")
    show_panel(
        query       = question,
        token_used  = tokens_used,
        strategy    = "multi-turn",
        prompt_size = prompt_size,
    )
    console.print(f"\n[bold green]Reply:[/bold green] {reply[:200]}{'…' if len(reply) > 200 else ''}\n")

────────────────────────────────────────────── Turn 1 — before call ───────────────────────────────────────────────

╭─ Pocket Agent  ·  What is a Python list comprehension? Give a one-line example. ─╮
│ Token Budget      ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  0 / 4,096 tokens             │
│                                                                                  │
│ Retrieved         (none)                                                         │
│ Strategy          multi-turn                                                     │
│ Prompt size       15 tokens                                                      │
╰──────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Turn 1 — after call ───────────────────────────────────────────────

╭─ Pocket Agent  ·  What is a Python list comprehension? Give a one-line example. ─╮
│ Token Budget      ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  105 / 4,096 tokens           │
│                                                                                  │
│ Retrieved         (none)                                                         │
│ Strategy          multi-turn                                                     │
│ Prompt size       15 tokens                                                      │
╰──────────────────────────────────────────────────────────────────────────────────╯

Reply: A Python list comprehension is a concise syntax for creating a new list by applying an expression to each 
item in an iterable (like a list, range, or generator), optionally filtering items with a cond…

...

────────────────────────────────────────────── Turn 4 — before call ───────────────────────────────────────────────

╭─ Pocket Agent  ·  Show me a minimal example where a shallow copy causes a bug. ─╮
│ Token Budget      ███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  401 / 4,096 tokens          │
│                                                                                 │
│ Retrieved         (none)                                                        │
│ Strategy          multi-turn                                                    │
│ Prompt size       346 tokens                                                    │
╰─────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Turn 4 — after call ───────────────────────────────────────────────

╭─ Pocket Agent  ·  Show me a minimal example where a shallow copy causes a bug. ─╮
│ Token Budget      █████░░░░░░░░░░░░░░░░░░░░░░░░░░░  680 / 4,096 tokens          │
│                                                                                 │
│ Retrieved         (none)                                                        │
│ Strategy          multi-turn                                                    │
│ Prompt size       346 tokens                                                    │
╰─────────────────────────────────────────────────────────────────────────────────╯

Reply: Here’s a minimal, real-world bug scenario: managing user sessions in a web app.

```python
# ❌ Bug: Using shallow copy (list.copy())
sessions = [
    {'user': 'alice', 'active': True},
    {'user': 'b…

### 2.3 What files actually cost

A coding agent loads source files to answer questions about them.
Let's measure what that costs in tokens.

We'll scan the files in `REPO_ROOT`, count their token cost, and show
how quickly a naive "load everything" strategy would exhaust the budget.

In [125]:
import os
from pathlib import Path
from rich.table import Table

# File extensions we'd want to load for a coding question
CODE_EXTENSIONS = {".py", ".js", ".ts", ".go", ".rs", ".java",
                   ".c", ".cpp", ".h", ".md", ".txt", ".yaml", ".toml", ".json"}

def scan_repo_costs(root: str) -> list[dict]:
    """
    Walk *root* and return a list of dicts:
        {"path": relative_path, "bytes": int, "tokens": int}
    for every source file found, sorted by token cost descending.
    """
    results = []
    for dirpath, dirnames, filenames in os.walk(root):
        # Skip hidden dirs and common noise
        dirnames[:] = [d for d in dirnames
                       if not d.startswith(".") and d not in
                       {"__pycache__", "node_modules", ".git", "venv", ".venv"}]
        for fname in filenames:
            full = Path(dirpath) / fname
            if full.suffix.lower() not in CODE_EXTENSIONS:
                continue
            try:
                text = full.read_text(errors="ignore")
            except OSError:
                continue
            results.append({
                "path":   str(full.relative_to(root)),
                "bytes":  len(text.encode()),
                "tokens": count_tokens(text),
            })
    return sorted(results, key=lambda x: x["tokens"], reverse=True)


file_costs = scan_repo_costs(REPO_ROOT)

# ── Rich table ──────────────────────────────────────────────────────────────
tbl = Table(title=f"File token costs in '{REPO_ROOT}'",
            show_lines=False, header_style="bold cyan")
tbl.add_column("File",        style="dim",       max_width=60)
tbl.add_column("Bytes",       justify="right")
tbl.add_column("Tokens",      justify="right")
tbl.add_column("% of budget", justify="right")

cumulative = 0
for f in file_costs[:20]:          # show top 20
    pct = f["tokens"] / TOKEN_BUDGET * 100
    cumulative += f["tokens"]
    cum_pct = cumulative / TOKEN_BUDGET * 100
    color = "green" if pct < 10 else ("yellow" if pct < 30 else "red")
    tbl.add_row(
        f["path"],
        f"{f['bytes']:,}",
        f"[{color}]{f['tokens']:,}[/{color}]",
        f"[{color}]{pct:.1f} %[/{color}]",
    )

console.print(tbl)

total_tokens = sum(f["tokens"] for f in file_costs)
console.print(
    f"\n[bold]Total across {len(file_costs)} files:[/bold] "
    f"[{'red' if total_tokens > TOKEN_BUDGET else 'green'}]{total_tokens:,} tokens[/]\n"
    f"Token budget: {TOKEN_BUDGET:,}\n"
    f"Fits in one prompt: [bold]{'NO ❌' if total_tokens > TOKEN_BUDGET else 'YES ✓'}[/bold]"
)

                           File token costs in '.'                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━┓
┃ File                                       ┃ Bytes ┃ Tokens ┃ % of budget ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━┩
│ README.md                                  │ 6,645 │  1,650 │      40.3 % │
│ sample_project/tests/test_formatter_gen.py │ 1,989 │    497 │      12.1 % │
│ AGENTS.md                                  │ 1,323 │    326 │       8.0 % │
│ sample_project/utils/parser.py             │ 1,221 │    303 │       7.4 % │
│ sample_project/tests/test_parser.py        │ 1,116 │    279 │       6.8 % │
│ sample_project/utils/validator.py          │ 1,044 │    261 │       6.4 % │
│ sample_project/utils/formatter.py          │   895 │    223 │       5.4 % │
│ sample_project/main.py                     │   694 │    173 │       4.2 % │
│ sample_project/tests/test_formatter.py     │   674 │    168 │       4.1 % │
└────────────────────────────────────────────┴───────┴────────┴─────────────┘

Total across 9 files: 3,880 tokens
Token budget: 4,096
Fits in one prompt: YES ✓

### 2.4 Hitting the wall — what over-budget looks like

Let's simulate what happens when a naive agent stuffs too much into one prompt.
We'll build a fake "load everything" prompt, measure it, and show the panel
turning red — without actually sending it to Ollama (no point wasting tokens
on a prompt that would be truncated or error).

In [126]:
def _simulate_overbudget() -> None:
    """
    Build a hypothetical prompt that exceeds TOKEN_BUDGET and show
    the panel at 50 %, 85 %, and 110 % capacity — no LLM call needed.
    """
    # Synthetic file list — imagine a mid-size Python project
    fake_files = [
        {"name": "src/parser.py",      "tokens": int(TOKEN_BUDGET * 0.18)},
        {"name": "src/compiler.py",    "tokens": int(TOKEN_BUDGET * 0.22)},
        {"name": "src/runtime.py",     "tokens": int(TOKEN_BUDGET * 0.20)},
        {"name": "tests/test_parser.py","tokens": int(TOKEN_BUDGET * 0.15)},
        {"name": "docs/architecture.md","tokens": int(TOKEN_BUDGET * 0.12)},
        {"name": "README.md",          "tokens": int(TOKEN_BUDGET * 0.08)},
        {"name": "setup.py",           "tokens": int(TOKEN_BUDGET * 0.05)},
    ]

    loaded_files = []
    accumulated  = 0
    query        = "Explain how the compiler hands off to the runtime"

    for i, f in enumerate(fake_files):
        accumulated += f["tokens"]
        loaded_files.append({"path": f["name"], "hot": True})

        pct = accumulated / TOKEN_BUDGET
        label = (
            "[green]comfortable[/green]" if pct < 0.6 else
            "[yellow]warning[/yellow]"   if pct < 0.85 else
            "[red]OVER BUDGET[/red]"
        )
        console.rule(f"After loading {i+1} file(s) — {label}")
        show_panel(
            query        = query,
            token_used   = accumulated,
            files        = loaded_files,
            strategy     = "naive load-all",
            prompt_size  = accumulated,
        )

        if accumulated > TOKEN_BUDGET:
            console.print(
                "\n[bold red]⚠  Prompt exceeds context window.[/bold red]\n"
                "Ollama will truncate the earliest messages — the agent may\n"
                "forget the system prompt or earlier file contents entirely.\n"
            )
            break

_simulate_overbudget()

────────────────────────────────────── After loading 1 file(s) — comfortable ──────────────────────────────────────

╭── Pocket Agent  ·  Explain how the compiler hands off to the runtime ──╮
│ Token Budget      █████░░░░░░░░░░░░░░░░░░░░░░░░░░░  737 / 4,096 tokens │
│                                                                        │
│ Retrieved          HOT    src/parser.py                                │
│                                                                        │
│ Strategy          naive load-all                                       │
│ Prompt size       737 tokens                                           │
╰────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────── After loading 2 file(s) — comfortable ──────────────────────────────────────

╭─── Pocket Agent  ·  Explain how the compiler hands off to the runtime ───╮
│ Token Budget      ████████████░░░░░░░░░░░░░░░░░░░░  1,638 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    src/parser.py                                  │
│                    HOT    src/compiler.py                                │
│                                                                          │
│ Strategy          naive load-all                                         │
│ Prompt size       1,638 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────── After loading 3 file(s) — comfortable ──────────────────────────────────────

...

╭─── Pocket Agent  ·  Explain how the compiler hands off to the runtime ───╮
│ Token Budget      ███████████████████████████░░░░░  3,562 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    src/parser.py                                  │
│                    HOT    src/compiler.py                                │
│                    HOT    src/runtime.py                                 │
│                    HOT    tests/test_parser.py                           │
│                    HOT    docs/architecture.md                           │
│                                                                          │
│ Strategy          naive load-all                                         │
│ Prompt size       3,562 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────── After loading 6 file(s) — OVER BUDGET ──────────────────────────────────────

╭─── Pocket Agent  ·  Explain how the compiler hands off to the runtime ───╮
│ Token Budget      ██████████████████████████████░░  3,889 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    src/parser.py                                  │
│                    HOT    src/compiler.py                                │
│                    HOT    src/runtime.py                                 │
│                    HOT    tests/test_parser.py                           │
│                    HOT    docs/architecture.md                           │
│                    HOT    README.md                                      │
│                                                                          │
│ Strategy          naive load-all                                         │
│ Prompt size       3,889 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────── After loading 7 file(s) — OVER BUDGET ──────────────────────────────────────

╭─── Pocket Agent  ·  Explain how the compiler hands off to the runtime ───╮
│ Token Budget      ███████████████████████████████░  4,093 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    src/parser.py                                  │
│                    HOT    src/compiler.py                                │
│                    HOT    src/runtime.py                                 │
│                    HOT    tests/test_parser.py                           │
│                    HOT    docs/architecture.md                           │
│                    HOT    README.md                                      │
│                    HOT    setup.py                                       │
│                                                                          │
│ Strategy          naive load-all                                         │
│ Prompt size       4,093 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

### 2.5 The three strategies — what's coming next

Every technique in Chapters 3–8 is a variation of one of these three responses
to budget pressure:

| Strategy | What it does | Where we build it |
|----------|-------------|-------------------|
| **Be selective** | Only load files relevant to the query | Ch 3 (manifest), Ch 4 (glob), Ch 5 (fuzzy), Ch 6 (grep) |
| **Be lazy** | Load file *summaries* first, full content only on demand | Ch 4 JIT reads |
| **Evict & compress** | Move cold context to a summary store; keep hot context small | Ch 7 microcompaction, Ch 8 semantic RAG |

The key insight: **a good agent spends tokens like a good programmer spends CPU** — only when necessary, and on the thing most likely to matter.

Chapter 3 introduces the first tool: an `AGENTS.md` manifest that tells the agent which files matter *before* it even looks at the file tree.

---
## Chapter 3 — Give it a Map

**Goal:** before the agent looks at a single file, hand it a curated index of the repo so it already knows what exists and what matters.

That index is `AGENTS.md` — a small markdown file you commit at the repo root.
It costs a fixed, known number of tokens on every call (cheap), and it dramatically reduces the search space for every retrieval strategy we build later.

**You will:**
- Create an `AGENTS.md` for this repo
- Write `load_manifest()` — reads the file, extracts mentioned paths
- Write `ask_with_manifest()` — prepends the manifest to every prompt
- See the panel report the manifest's token cost before any file is loaded

### 3.1 What goes in AGENTS.md?

An `AGENTS.md` is just a markdown file. It has no required schema — the agent
reads it as plain text. The convention is:

- **Entry points** — where execution starts
- **Key modules** — what each important file does in one line
- **Ownership sections** — "questions about X → look in Y"
- **Off-limits** — generated files, build artefacts the agent should skip

Keep it under ~400 tokens. If it grows larger than that, it starts eating the
budget it was meant to protect.

The cell below uses `%%writefile` to create `AGENTS.md` on disk.
`%%writefile path` is a Jupyter magic: instead of running the cell,
it saves the cell body verbatim to the given path. The file will live next to
this notebook in `REPO_ROOT`.

In [127]:
%%writefile AGENTS.md
# AGENTS.md — Pocket Agent project map

## What this repo is
A Jupyter notebook tutorial that builds a local coding agent step by step.
Each chapter adds one capability. The notebook IS the source of truth.

## Entry point
- `pocket_agent.ipynb` — the entire project lives here

## Chapter guide
| Chapter | Topic | Key concepts defined |
|---------|-------|----------------------|
| 1 | Hello Ollama | `chat()`, `show_panel()`, `ping_ollama()` |
| 2 | Context budget | `count_tokens()`, `scan_repo_costs()` |
| 3 | Manifest | `load_manifest()`, `ask_with_manifest()` |
| 4 | Glob + JIT reads | `glob_files()`, `jit_read()` |
| 5 | Fuzzy scoring | `score_files()` |
| 6 | Grep | `grep_repo()` |
| 7 | Microcompaction | `compact()`, hot/cold store |
| 8 | Semantic RAG | `embed()`, `retrieve()` |
| 9 | Full pipeline | `run()` |
| 9b | Web UI | FastAPI + WebSocket server |
| 10 | Write + diff | `write_file()`, `make_diff()` |
| 11 | Agent loop | `agent_loop()` |
| 12 | Test generation | `generate_tests()` |

## Off-limits (never load these)
- `.git/` — git internals
- `__pycache__/` — compiled bytecode
- `*.ipynb_checkpoints/` — Jupyter autosave noise

## Questions about token budgeting → Chapter 2
## Questions about retrieval strategy → Chapters 4–8
## Questions about the agent loop → Chapter 11

Overwriting AGENTS.md


### 3.2 `load_manifest()` — read the map

`load_manifest()` does two things:
1. Reads `AGENTS.md` as plain text (to inject into the prompt)
2. Extracts any file paths mentioned in it (for later retrieval stages to use as hints)

The path extraction is intentionally simple — a regex that finds things
that look like `path/to/file.ext`. False positives are fine; this is
advisory, not authoritative.

In [128]:
import re

MANIFEST_FILENAME = "AGENTS.md"

def load_manifest(repo_root: str = REPO_ROOT) -> dict:
    """
    Read AGENTS.md from *repo_root*.

    Returns a dict:
        {
          "text":   str,        # full file content, ready to inject into a prompt
          "tokens": int,        # estimated token cost
          "paths":  list[str],  # file paths mentioned in the manifest
          "found":  bool,       # False if the file doesn't exist
        }
    """
    manifest_path = Path(repo_root) / MANIFEST_FILENAME

    if not manifest_path.exists():
        return {"text": "", "tokens": 0, "paths": [], "found": False}

    text = manifest_path.read_text(errors="ignore")

    # Extract things that look like file paths: word chars, slashes, dots
    # e.g.  pocket_agent.ipynb  src/parser.py  docs/architecture.md
    paths = re.findall(r'\b[\w./\-]+\.[\w]+\b', text)
    # Deduplicate while preserving order
    seen, unique_paths = set(), []
    for p in paths:
        if p not in seen:
            seen.add(p)
            unique_paths.append(p)

    return {
        "text":   text,
        "tokens": count_tokens(text),
        "paths":  unique_paths,
        "found":  True,
    }


manifest = load_manifest()
print(f"Manifest found : {manifest['found']}")
print(f"Token cost     : {manifest['tokens']}  ({manifest['tokens']/TOKEN_BUDGET*100:.1f}% of budget)")
print(f"Paths mentioned: {manifest['paths'][:8]}")

Manifest found : True
Token cost     : 326  (8.0% of budget)
Paths mentioned: ['AGENTS.md', 'pocket_agent.ipynb']


### 3.3 `ask_with_manifest()` — the manifest-aware query

Every prompt the agent sends from now on follows this structure:

```
[SYSTEM]
You are a coding assistant. Here is the project map:
<AGENTS.md contents>

[USER]
<query>
```

The manifest is injected once, costs a fixed number of tokens, and gives
the model the project layout before it has to reason about anything.

In [129]:
def ask_with_manifest(
    query:     str,
    repo_root: str = REPO_ROOT,
    files:     list[dict] | None = None,
) -> tuple[str, int]:
    """
    Send *query* to Ollama with the project manifest prepended as a system prompt.

    Parameters
    ----------
    query     : the user's question
    repo_root : where to find AGENTS.md
    files     : already-loaded file dicts {"path", "content", "hot"}
                their content is appended after the manifest

    Returns (reply_text, tokens_used).
    """
    manifest = load_manifest(repo_root)
    files    = files or []

    # Build system prompt
    if manifest["found"]:
        system_content = (
            "You are a coding assistant with access to the project map below.\n"
            "Use it to understand the codebase structure before answering.\n\n"
            f"--- PROJECT MAP (AGENTS.md) ---\n{manifest['text']}\n---"
        )
    else:
        system_content = "You are a coding assistant."

    # Append any loaded file contents
    file_block = ""
    for f in files:
        file_block += f"\n\n--- FILE: {f['path']} ---\n{f.get('content', '')}"

    messages = [
        {"role": "system",  "content": system_content},
        {"role": "user",    "content": query + file_block},
    ]

    prompt_size = count_tokens(system_content + query + file_block)

    show_panel(
        query        = query,
        token_used   = prompt_size,
        files        = [{"path": f["path"], "hot": f.get("hot", True)} for f in files],
        strategy     = "manifest",
        prompt_size  = prompt_size,
    )

    reply, tokens_used = chat(messages)
    return reply, tokens_used

### 3.4 Try it

Ask a question about the project. The manifest is in the prompt so the model
already knows the chapter structure before it replies.

In [130]:
query = "Which chapter should I look at to understand how retrieval works?"

console.rule("[bold]Chapter 3 — manifest-guided query[/bold]")
reply, tokens_used = ask_with_manifest(query)

console.rule("[bold]After call[/bold]")
show_panel(
    query       = query,
    token_used  = tokens_used,
    strategy    = "manifest",
    prompt_size = count_tokens(query),
)
console.print(f"\n[bold green]Response:[/bold green]\n{reply}")

──────────────────────────────────────── Chapter 3 — manifest-guided query ────────────────────────────────────────

╭─ Pocket Agent  ·  Which chapter should I look at to understand how retrieval works? ─╮
│ Token Budget      ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  383 / 4,096 tokens               │
│                                                                                      │
│ Retrieved         (none)                                                             │
│ Strategy          manifest                                                           │
│ Prompt size       383 tokens                                                         │
╰──────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── After call ────────────────────────────────────────────────────

╭─ Pocket Agent  ·  Which chapter should I look at to understand how retrieval works? ─╮
│ Token Budget      ████░░░░░░░░░░░░░░░░░░░░░░░░░░░░  565 / 4,096 tokens               │
│                                                                                      │
│ Retrieved         (none)                                                             │
│ Strategy          manifest                                                           │
│ Prompt size       16 tokens                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Response:
To understand how retrieval works, look at **Chapter 8: Semantic RAG**, which covers:

- `embed()` — how content is turned into vector embeddings  
- `retrieve()` — how relevant chunks are fetched based on semantic similarity

For background context (like how files are scanned and prepared for retrieval), you may also want to briefly 
review:

- **Chapter 4**: `glob_files()`, `jit_read()` — file discovery and just-in-time loading  
- **Chapter 7**: `compact()` and hot/cold store — how context is preprocessed for efficiency  

But the core retrieval logic is explicitly in **Chapter 8**.

---
## Chapter 4 — Glob + JIT Reads

**Goal:** navigate a file tree without bulk-loading it.

Chapter 3 gave the agent a curated map. But what about files the map doesn't
mention, or repos where no `AGENTS.md` exists? The agent needs to *find* files
on its own — without reading them all.

The answer is a two-phase approach:

1. **Glob** — list every file that *could* be relevant (filenames only, no content).  
   This is essentially free: no tokens spent yet.
2. **JIT read** — load each file's content *just in time*, one at a time,  
   and stop when the token budget would tip past a safe threshold.

Files that were found but not loaded appear as **COLD** in the panel.
Files whose content is in the prompt are **HOT**.

**You will:**
- Create a small sample project to give glob something to work with
- Write `glob_files()` — match file paths against a pattern
- Write `jit_read()` — load one file on demand
- Write `budget_load()` — greedily load HOT files until budget threshold
- See the panel split between HOT and COLD for the first time

### 4.1 Sample project

Our repo only has one notebook and `AGENTS.md` — not enough to demonstrate
file navigation. The cells below use `%%writefile` to create a small fake
Python project under `sample_project/`.

Run each cell once; the files persist on disk for all later chapters.

In [131]:
from typing import NamedTuple


class RunResult(NamedTuple):
    reply:       str
    strategy:    str
    files:       list[dict]   # full list, HOT and COLD
    tokens_used: int
    compact_log: list[str]    # empty if compaction didn't fire


def run(
    query:     str,
    repo_root: str  = REPO_ROOT,
    strategy:  str  = "auto",   # "auto" | "fuzzy" | "grep" | "semantic"
    top_k:     int  = 8,
) -> RunResult:
    """
    Full retrieval + generation pipeline.

    Parameters
    ----------
    query     : the user's question
    repo_root : path to the repository to search
    strategy  : retrieval strategy; "auto" lets pick_strategy() decide
    top_k     : max candidates to consider before budget_load()
    """
    # ── 1. Manifest ─────────────────────────────────────────────────────────
    manifest     = load_manifest(repo_root)    # from the target repo, not REPO_ROOT
    manifest_tok = manifest["tokens"]

    # ── 2. Strategy selection ────────────────────────────────────────────────
    strat = pick_strategy(query) if strategy == "auto" else strategy

    # ── 3. Retrieval ─────────────────────────────────────────────────────────
    if strat == "grep":
        candidates = grep_rank(query, repo_root=repo_root)
        # Append fuzzy extras for files with zero grep hits
        found_paths = {f["path"] for f in candidates}
        extras = [f for f in rank_files(glob_files("*.py", repo_root=repo_root), query)
                  if f["path"] not in found_paths]
        candidates = (candidates + extras)[:top_k]

    elif strat == "semantic":
        candidates = semantic_retrieve(query, repo_root=repo_root, top_k=top_k)

    else:   # fuzzy
        candidates = rank_files(glob_files("*.py", repo_root=repo_root), query)[:top_k]

    # ── 4. Budget-aware JIT load ─────────────────────────────────────────────
    loaded = budget_load(candidates, already_used=manifest_tok, repo_root=repo_root)

    # ── 5. Compaction (if needed) ─────────────────────────────────────────────
    hot_tok = sum(f.get("tokens", 0) for f in loaded if f["hot"])
    total   = manifest_tok + hot_tok + count_tokens(query)
    loaded, total, compact_log = compact(loaded, query, total)

    # ── 6. Show panel ─────────────────────────────────────────────────────────
    hot_files = [f for f in loaded if f["hot"]]
    show_panel(
        query        = query,
        token_used   = total,
        files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded],
        strategy     = strat,
        prompt_size  = total,
    )

    # ── 7. Generate reply ─────────────────────────────────────────────────────
    reply, tokens_used = ask_with_manifest(query, repo_root=repo_root, files=hot_files)

    return RunResult(
        reply        = reply,
        strategy     = strat,
        files        = loaded,
        tokens_used  = tokens_used,
        compact_log  = compact_log,
    )

In [132]:
%%writefile sample_project/main.py
"""Entry point for the JSON-to-CSV converter tool."""

from utils.parser import parse_json
from utils.formatter import to_csv
from utils.validator import validate_schema
import sys


def main(input_path: str, output_path: str, schema_path: str | None = None) -> None:
    with open(input_path) as f:
        raw = f.read()

    records = parse_json(raw)

    if schema_path:
        with open(schema_path) as f:
            schema = f.read()
        validate_schema(records, schema)

    csv_text = to_csv(records)

    with open(output_path, "w") as f:
        f.write(csv_text)

    print(f"Wrote {len(records)} records to {output_path}")


if __name__ == "__main__":
    main(*sys.argv[1:])

Overwriting sample_project/main.py


In [133]:
%%writefile sample_project/utils/parser.py
"""Parse a JSON string into a list of flat dicts."""

import json


def parse_json(raw: str) -> list[dict]:
    """
    Accept a JSON string that is either:
    - a list of objects   → returned as-is
    - a single object     → wrapped in a list
    - a list of scalars   → each scalar wrapped as {"value": scalar}

    Raises ValueError on malformed input.
    """
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON: {exc}") from exc

    if isinstance(data, list):
        return [_flatten(item) if isinstance(item, dict) else {"value": item}
                for item in data]
    if isinstance(data, dict):
        return [_flatten(data)]

    raise ValueError(f"Expected a JSON object or array, got {type(data).__name__}")


def _flatten(obj: dict, prefix: str = "", sep: str = ".") -> dict:
    """Recursively flatten nested dicts: {"a": {"b": 1}} → {"a.b": 1}."""
    result = {}
    for key, value in obj.items():
        full_key = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            result.update(_flatten(value, full_key, sep))
        else:
            result[full_key] = value
    return result

Overwriting sample_project/utils/parser.py


In [134]:
%%writefile sample_project/utils/formatter.py
"""Format a list of flat dicts as CSV text."""

import csv
import io


def to_csv(records: list[dict], delimiter: str = ",") -> str:
    """
    Convert *records* (list of flat dicts) to a CSV string.

    All keys across all records are used as headers.
    Missing values are rendered as empty strings.
    """
    if not records:
        return ""

    # Collect all headers preserving first-seen order
    headers: list[str] = []
    seen: set[str] = set()
    for rec in records:
        for key in rec:
            if key not in seen:
                headers.append(key)
                seen.add(key)

    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=headers,
                            delimiter=delimiter, extrasaction="ignore")
    writer.writeheader()
    for rec in records:
        writer.writerow({h: rec.get(h, "") for h in headers})

    return buf.getvalue()

Overwriting sample_project/utils/formatter.py


In [135]:
%%writefile sample_project/utils/validator.py
"""Validate a list of records against a simple JSON schema."""

import json


def validate_schema(records: list[dict], schema_raw: str) -> None:
    """
    Validate each record against *schema_raw* (a JSON object mapping
    field names to expected types: "string", "number", "boolean").

    Raises TypeError on the first violation found.
    """
    schema: dict[str, str] = json.loads(schema_raw)
    type_map = {"string": str, "number": (int, float), "boolean": bool}

    for i, record in enumerate(records):
        for field, expected_type_name in schema.items():
            if field not in record:
                raise TypeError(
                    f"Record {i}: missing required field '{field}'"
                )
            expected = type_map.get(expected_type_name)
            if expected and not isinstance(record[field], expected):
                raise TypeError(
                    f"Record {i}: field '{field}' expected {expected_type_name}, "
                    f"got {type(record[field]).__name__}"
                )

Overwriting sample_project/utils/validator.py


In [136]:
%%writefile sample_project/tests/test_parser.py
"""Unit tests for utils/parser.py."""

import pytest
from utils.parser import parse_json, _flatten


class TestParseJson:
    def test_list_of_dicts(self):
        result = parse_json('[{"a": 1}, {"a": 2}]')
        assert result == [{"a": 1}, {"a": 2}]

    def test_single_dict_wrapped(self):
        result = parse_json('{"x": 10}')
        assert result == [{"x": 10}]

    def test_list_of_scalars(self):
        result = parse_json('[1, 2, 3]')
        assert result == [{"value": 1}, {"value": 2}, {"value": 3}]

    def test_invalid_json_raises(self):
        with pytest.raises(ValueError, match="Invalid JSON"):
            parse_json("{not valid}")

    def test_unexpected_type_raises(self):
        with pytest.raises(ValueError, match="Expected"):
            parse_json('"just a string"')


class TestFlatten:
    def test_nested_dict(self):
        assert _flatten({"a": {"b": 1}}) == {"a.b": 1}

    def test_deeply_nested(self):
        assert _flatten({"a": {"b": {"c": 42}}}) == {"a.b.c": 42}

    def test_flat_dict_unchanged(self):
        assert _flatten({"x": 1, "y": 2}) == {"x": 1, "y": 2}

Overwriting sample_project/tests/test_parser.py


In [137]:
%%writefile sample_project/tests/test_formatter.py
"""Unit tests for utils/formatter.py."""

from utils.formatter import to_csv


def test_empty_input():
    assert to_csv([]) == ""


def test_single_record():
    result = to_csv([{"name": "Alice", "age": 30}])
    lines = result.strip().splitlines()
    assert lines[0] == "name,age"
    assert lines[1] == "Alice,30"


def test_missing_fields_become_empty():
    records = [{"a": 1, "b": 2}, {"a": 3}]
    result = to_csv(records)
    lines = result.strip().splitlines()
    assert lines[0] == "a,b"
    assert lines[2] == "3,"


def test_header_order_follows_first_seen():
    records = [{"z": 1, "a": 2}]
    result = to_csv(records)
    assert result.startswith("z,a")

Overwriting sample_project/tests/test_formatter.py


### 4.2 `glob_files()` — list without loading

`glob_files()` returns a list of matching file paths and their sizes —
but **no file content**. Zero tokens spent so far.

The `pattern` argument follows Python's `fnmatch` syntax:  
`*.py` matches any `.py` file, `utils/*.py` matches only files directly in `utils/`.

In [138]:
import fnmatch

SKIP_DIRS = {".git", "__pycache__", "node_modules", ".venv", "venv",
             ".ipynb_checkpoints", ".mypy_cache", ".pytest_cache"}


def glob_files(
    pattern:   str,
    repo_root: str = REPO_ROOT,
) -> list[dict]:
    """
    Walk *repo_root* and return every file whose name matches *pattern*.

    Returns a list of dicts:
        {"path": str,   # relative to repo_root
         "bytes": int,  # file size — no content loaded
         "hot":   bool} # always False at this stage
    Sorted by path for deterministic ordering.
    """
    matches = []
    root = Path(repo_root)

    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".")]
        for fname in filenames:
            if fnmatch.fnmatch(fname, pattern):
                full = Path(dirpath) / fname
                try:
                    size = full.stat().st_size
                except OSError:
                    continue
                matches.append({
                    "path":  str(full.relative_to(root)),
                    "bytes": size,
                    "hot":   False,
                })

    return sorted(matches, key=lambda x: x["path"])


# ── Demo ────────────────────────────────────────────────────────────────────
found = glob_files("*.py", repo_root="sample_project")
print(f"Found {len(found)} Python files:\n")
for f in found:
    print(f"  {f['path']:45s}  {f['bytes']:>5} bytes")

Found 7 Python files:

  main.py                                          694 bytes
  tests/test_formatter.py                          674 bytes
  tests/test_formatter_gen.py                     1989 bytes
  tests/test_parser.py                            1116 bytes
  utils/formatter.py                               895 bytes
  utils/parser.py                                 1221 bytes
  utils/validator.py                              1044 bytes


### 4.3 `jit_read()` — load one file on demand

`jit_read()` takes a path from the glob list and reads its content.
It returns the same dict shape, extended with `"content"` and `"tokens"`,
and flips `"hot"` to `True`.

In [139]:
def jit_read(file_meta: dict, repo_root: str = REPO_ROOT) -> dict:
    """
    Read the content of one file described by *file_meta* (a glob result dict).

    Returns a new dict with the same keys plus:
        "content": str   — full file text
        "tokens":  int   — estimated token cost
        "hot":     True  — this file is now in the prompt
    """
    full_path = Path(repo_root) / file_meta["path"]
    try:
        content = full_path.read_text(errors="ignore")
    except OSError as exc:
        content = f"[error reading file: {exc}]"

    return {
        **file_meta,
        "content": content,
        "tokens":  count_tokens(content),
        "hot":     True,
    }


# ── Demo: read one file ─────────────────────────────────────────────────────
sample_meta = glob_files("parser.py", repo_root="sample_project")[0]
loaded      = jit_read(sample_meta, repo_root="sample_project")

print(f"Path   : {loaded['path']}")
print(f"Tokens : {loaded['tokens']}")
print(f"Hot    : {loaded['hot']}")
print(f"\nFirst 3 lines:\n{''.join(loaded['content'].splitlines(keepends=True)[:3])}")

Path   : utils/parser.py
Tokens : 303
Hot    : True

First 3 lines:
"""Parse a JSON string into a list of flat dicts."""

import json



### 4.4 `budget_load()` — greedy HOT/COLD split

Now we combine glob and JIT read.
`budget_load()` takes a list of glob results, reads them one by one,
and stops before the token budget hits a safety threshold.
Files it couldn't fit are kept in the list as COLD.

The threshold is 0.7 (70 % of budget) — leaving room for the manifest,
the query, and the model's reply.

In [140]:
HOT_THRESHOLD = 0.70   # stop loading when prompt would exceed 70 % of budget


def budget_load(
    candidates: list[dict],
    already_used: int      = 0,
    repo_root:    str      = REPO_ROOT,
    threshold:    float    = HOT_THRESHOLD,
) -> list[dict]:
    """
    JIT-read files from *candidates* until adding the next one would push
    the running token total past *threshold* × TOKEN_BUDGET.

    Returns the full candidate list with:
      - loaded files marked  hot=True  and populated with "content"/"tokens"
      - unloaded files kept  hot=False (no content key added)
    """
    budget_limit = int(TOKEN_BUDGET * threshold)
    used         = already_used
    result       = []

    for meta in candidates:
        headroom = budget_limit - used
        # Peek at size without reading: use byte count as a cheap proxy
        estimated = meta["bytes"] // 4
        if estimated > headroom:
            result.append({**meta, "hot": False})
            continue

        loaded = jit_read(meta, repo_root=repo_root)
        if used + loaded["tokens"] > budget_limit:
            result.append({**meta, "hot": False})
        else:
            used += loaded["tokens"]
            result.append(loaded)

    return result


# ── Demo: load sample_project with a tiny artificial budget ─────────────────
candidates = glob_files("*.py", repo_root="sample_project")
loaded_files = budget_load(candidates, repo_root="sample_project")

hot  = [f for f in loaded_files if f["hot"]]
cold = [f for f in loaded_files if not f["hot"]]

console.rule("[bold]budget_load() result[/bold]")
show_panel(
    query        = "How does the project handle nested JSON objects?",
    token_used   = sum(f.get("tokens", 0) for f in hot),
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "glob + JIT",
    prompt_size  = sum(f.get("tokens", 0) for f in hot),
)
print(f"\nHOT ({len(hot)} files, {sum(f['tokens'] for f in hot):,} tokens):")
for f in hot:
    print(f"  ✓ {f['path']}")
print(f"\nCOLD ({len(cold)} files — found but not loaded):")
for f in cold:
    print(f"  ✗ {f['path']}")

────────────────────────────────────────────── budget_load() result ───────────────────────────────────────────────

╭─── Pocket Agent  ·  How does the project handle nested JSON objects? ────╮
│ Token Budget      ██████████████░░░░░░░░░░░░░░░░░░  1,904 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    main.py                                        │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/formatter.py                             │
│                    HOT    utils/parser.py                                │
│                    HOT    utils/validator.py                             │
│                                                                          │
│ Strategy          glob + JIT                                             │
│ Prompt size       1,904 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯


HOT (7 files, 1,904 tokens):
  ✓ main.py
  ✓ tests/test_formatter.py
  ✓ tests/test_formatter_gen.py
  ✓ tests/test_parser.py
  ✓ utils/formatter.py
  ✓ utils/parser.py
  ✓ utils/validator.py

COLD (0 files — found but not loaded):


### 4.5 Try it end-to-end

Ask a question about the sample project.
The agent globs for Python files, JIT-loads what fits in the budget,
and sends the HOT files plus the manifest to Ollama.

In [141]:
query      = "How does the parser handle a JSON array of plain numbers like [1, 2, 3]?"
repo       = "sample_project"

# Phase 1: glob — free, no tokens spent
candidates   = glob_files("*.py", repo_root=repo)

# Phase 2: budget-aware JIT load
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files    = [f for f in loaded_files if f["hot"]]
prompt_size  = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

console.rule("[bold]Chapter 4 — before LLM call[/bold]")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "glob + JIT",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

console.rule("[bold]After call[/bold]")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "glob + JIT",
    prompt_size = prompt_size,
)
console.print(f"\n[bold green]Response:[/bold green]\n{reply}")

─────────────────────────────────────────── Chapter 4 — before LLM call ───────────────────────────────────────────

╭─ Pocket Agent  ·  How does the parser handle a JSON array of plain numbers like [1, 2, 3… ─╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  1,922 / 4,096 tokens                   │
│                                                                                            │
│ Retrieved          HOT    main.py                                                          │
│                    HOT    tests/test_formatter.py                                          │
│                    HOT    tests/test_formatter_gen.py                                      │
│                    HOT    tests/test_parser.py                                             │
│                    HOT    utils/formatter.py                                               │
│                    HOT    utils/parser.py                                                  │
│                    HOT    utils/validator.py                                               │
│                                                                                            │
│ Strategy          glob + JIT                                                               │
│ Prompt size       1,922 tokens                                                             │
╰────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Pocket Agent  ·  How does the parser handle a JSON array of plain numbers like [1, 2, 3… ─╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  1,992 / 4,096 tokens                   │
│                                                                                            │
│ Retrieved          HOT    main.py                                                          │
│                    HOT    tests/test_formatter.py                                          │
│                    HOT    tests/test_formatter_gen.py                                      │
│                    HOT    tests/test_parser.py                                             │
│                    HOT    utils/formatter.py                                               │
│                    HOT    utils/parser.py                                                  │
│                    HOT    utils/validator.py                                               │
│                                                                                            │
│ Strategy          manifest                                                                 │
│ Prompt size       1,992 tokens                                                             │
╰────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── After call ────────────────────────────────────────────────────

╭─ Pocket Agent  ·  How does the parser handle a JSON array of plain numbers like [1, 2, 3… ─╮
│ Token Budget      ██████████████████░░░░░░░░░░░░░░  2,376 / 4,096 tokens                   │
│                                                                                            │
│ Retrieved          HOT    main.py                                                          │
│                    HOT    tests/test_formatter.py                                          │
│                    HOT    tests/test_formatter_gen.py                                      │
│                    HOT    tests/test_parser.py                                             │
│                    HOT    utils/formatter.py                                               │
│                    HOT    utils/parser.py                                                  │
│                    HOT    utils/validator.py                                               │
│                                                                                            │
│ Strategy          glob + JIT                                                               │
│ Prompt size       1,922 tokens                                                             │
╰────────────────────────────────────────────────────────────────────────────────────────────╯

Response:
Looking at the code, here's how the parser handles a JSON array of plain numbers like `[1, 2, 3]`:

1. **In `utils/parser.py`**: The `parse_json()` function processes the input:

   ```python
   def parse_json(raw: str) -> list:
       try:
           data = json.loads(raw)  # Parses "[1, 2, 3]" → [1, 2, 3]
       except json.JSONDecodeError as exc:
           raise ValueError(f"Invalid JSON: {exc}") from exc

       if isinstance(data, list):
           return [_flatten(item) if isinstance(item, dict) else {"value": item}
                   for item in data]
   ```

   Since `[1, 2, 3]` is a list, it enters the `if isinstance(data, list):` branch.

2. **For each item in the list**: The list comprehension checks if each item is a dict:
   - `1` is not a dict → `{"value": 1}`
   - `2` is not a dict → `{"value": 2}`
   - `3` is not a dict → `{"value": 3}`

3. **Result**: Returns `[{"value": 1}, {"value": 2}, {"value": 3}]`

This behavior is explicitly tested in `tests/test_parser.py`:

```python
def test_list_of_scalars(self):
    result = parse_json('[1, 2, 3]')
    assert result == [{"value": 1}, {"value": 2}, {"value": 3}]
```

So the parser converts a JSON array of plain numbers into a list of dictionaries, where each number becomes the 
value of a `"value"` key in a separate dictionary.

---
## Chapter 5 — Fuzzy Scoring

**Goal:** rank retrieved files before loading them, so budget_load() always drops the *least* relevant files when it runs out of room.

Chapter 4's glob returns files in alphabetical order.
If the budget fills up, it drops whatever comes last alphabetically — which may be the most relevant file.
Fuzzy scoring fixes this by sorting candidates by relevance *before* the JIT load loop.

Scoring works entirely on **file paths** — no content is read.
That keeps it free (no tokens, no disk I/O beyond what glob already did).

**Signals used:**
| Signal | Weight |
|--------|--------|
| Exact query word in filename stem | high |
| Fuzzy match between query word and filename stem | medium |
| Query word appears in a directory component | low |
| File is a test file, query doesn't mention tests | penalty ×0.5 |

**You will:**
- Write `tokenize_query()` — normalise a query into scoreable terms
- Write `score_file()` — score one file against the term list
- Write `rank_files()` — sort a candidate list by score descending
- Replace the alphabetical glob order with ranked order and see the panel change

### 5.1 `tokenize_query()` — extract scoreable terms

We strip stop words and short tokens so scores aren't diluted by
words like "the", "a", "how", "does".

In [142]:
import difflib

_STOP_WORDS = {
    "a", "an", "the", "is", "in", "it", "of", "to", "do", "does",
    "how", "what", "why", "when", "where", "which", "who", "for",
    "and", "or", "but", "not", "with", "this", "that", "are", "was",
    "i", "me", "my", "we", "our", "you", "your",
}


def tokenize_query(query: str) -> list[str]:
    """
    Lowercase, split on non-alphanumeric chars, remove stop words and
    tokens shorter than 3 characters.

    "How does the formatter handle missing fields?"
    → ["formatter", "handle", "missing", "fields"]
    """
    words = re.split(r"[^a-zA-Z0-9]+", query.lower())
    return [w for w in words if len(w) >= 3 and w not in _STOP_WORDS]


# Quick check
print(tokenize_query("How does the formatter handle missing fields?"))
print(tokenize_query("Where is the CSV delimiter configured?"))

['formatter', 'handle', 'missing', 'fields']
['csv', 'delimiter', 'configured']


### 5.2 `score_file()` — score one file against a query

`difflib.SequenceMatcher` gives us a similarity ratio between 0 and 1
with no extra dependencies — it's in the Python standard library.

In [143]:
def score_file(meta: dict, query_terms: list[str]) -> float:
    """
    Score *meta* (a glob result dict) against *query_terms*.

    Higher = more relevant.  Returns 0.0 for an empty term list.
    No file content is read — scoring is path-only.
    """
    if not query_terms:
        return 0.0

    path  = meta["path"].lower()
    parts = Path(path).parts          # ("utils", "parser.py")
    stem  = Path(path).stem           # "parser"
    dirs  = parts[:-1]                # ("utils",)

    score = 0.0
    for term in query_terms:
        # ── Filename stem ────────────────────────────────────────────
        if term == stem:
            score += 3.0                          # exact match
        elif term in stem or stem in term:
            score += 1.5                          # substring match
        else:
            ratio = difflib.SequenceMatcher(None, term, stem).ratio()
            score += ratio * 1.0                  # fuzzy match

        # ── Directory components ──────────────────────────────────────
        for d in dirs:
            if term == d:
                score += 1.0
            elif term in d:
                score += 0.5
            else:
                ratio = difflib.SequenceMatcher(None, term, d).ratio()
                score += ratio * 0.3

    # ── Test-file penalty ─────────────────────────────────────────────
    if "test" in path and "test" not in query_terms:
        score *= 0.5

    return round(score, 4)


# ── Sanity check ─────────────────────────────────────────────────────────────
_candidates = glob_files("*.py", repo_root="sample_project")
_terms      = tokenize_query("How does the formatter handle missing fields?")

for f in _candidates:
    print(f"{score_file(f, _terms):.4f}  {f['path']}")

1.4531  main.py
1.1723  tests/test_formatter.py
1.1713  tests/test_formatter_gen.py
0.6711  tests/test_parser.py
4.0194  utils/formatter.py
1.4149  utils/parser.py
1.2416  utils/validator.py


### 5.3 `rank_files()` — sort the candidate list

A thin wrapper that applies `score_file()` to every candidate and
returns them sorted highest-score-first.
Files with a score of 0 are kept — they're still candidates,
just lowest priority.

In [144]:
def rank_files(candidates: list[dict], query: str) -> list[dict]:
    """
    Return *candidates* sorted by relevance to *query*, highest first.
    Adds a "score" key to each dict.
    """
    terms = tokenize_query(query)
    scored = [
        {**c, "score": score_file(c, terms)}
        for c in candidates
    ]
    return sorted(scored, key=lambda x: x["score"], reverse=True)


# ── Show ranked order vs alphabetical ────────────────────────────────────────
query      = "How does the formatter handle missing fields?"
candidates = glob_files("*.py", repo_root="sample_project")
ranked     = rank_files(candidates, query)

tbl = Table(title="Ranked candidates", header_style="bold cyan", show_lines=False)
tbl.add_column("Rank", justify="right", style="dim")
tbl.add_column("Score", justify="right")
tbl.add_column("File")

for i, f in enumerate(ranked, 1):
    color = "green" if i == 1 else ("yellow" if i <= 3 else "dim")
    tbl.add_row(str(i), f"[{color}]{f['score']:.4f}[/{color}]", f["path"])

console.print(tbl)

               Ranked candidates               
┏━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Rank ┃  Score ┃ File                        ┃
┡━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    1 │ 4.0194 │ utils/formatter.py          │
│    2 │ 1.4531 │ main.py                     │
│    3 │ 1.4149 │ utils/parser.py             │
│    4 │ 1.2416 │ utils/validator.py          │
│    5 │ 1.1723 │ tests/test_formatter.py     │
│    6 │ 1.1713 │ tests/test_formatter_gen.py │
│    7 │ 0.6711 │ tests/test_parser.py        │
└──────┴────────┴─────────────────────────────┘

### 5.4 Try it — ranked glob + JIT

Same query as Chapter 4, but now candidates are ranked before budget_load()
sees them. The first file that turns COLD will be the least relevant, not a
random alphabetical casualty.

In [145]:
query = "How does the formatter handle missing fields?"
repo  = "sample_project"

# Phase 1: glob
candidates = glob_files("*.py", repo_root=repo)

# Phase 2: rank  ← new
ranked = rank_files(candidates, query)

# Phase 3: budget-aware JIT load (now works on ranked order)
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(ranked, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

console.rule("[bold]Chapter 5 — fuzzy-ranked before LLM call[/bold]")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "glob + fuzzy + JIT",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

console.rule("[bold]After call[/bold]")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "glob + fuzzy + JIT",
    prompt_size = prompt_size,
)
console.print(f"\n[bold green]Response:[/bold green]\n{reply}")

──────────────────────────────────── Chapter 5 — fuzzy-ranked before LLM call ─────────────────────────────────────

╭───── Pocket Agent  ·  How does the formatter handle missing fields? ─────╮
│ Token Budget      ██████████████░░░░░░░░░░░░░░░░░░  1,915 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/formatter.py                             │
│                    HOT    main.py                                        │
│                    HOT    utils/parser.py                                │
│                    HOT    utils/validator.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    tests/test_parser.py                           │
│                                                                          │
│ Strategy          glob + fuzzy + JIT                                     │
│ Prompt size       1,915 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

╭───── Pocket Agent  ·  How does the formatter handle missing fields? ─────╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  1,986 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/formatter.py                             │
│                    HOT    main.py                                        │
│                    HOT    utils/parser.py                                │
│                    HOT    utils/validator.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    tests/test_parser.py                           │
│                                                                          │
│ Strategy          manifest                                               │
│ Prompt size       1,986 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── After call ────────────────────────────────────────────────────

╭───── Pocket Agent  ·  How does the formatter handle missing fields? ─────╮
│ Token Budget      ███████████████████░░░░░░░░░░░░░  2,524 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/formatter.py                             │
│                    HOT    main.py                                        │
│                    HOT    utils/parser.py                                │
│                    HOT    utils/validator.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    tests/test_parser.py                           │
│                                                                          │
│ Strategy          glob + fuzzy + JIT                                     │
│ Prompt size       1,915 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

Response:
The formatter handles **missing fields by replacing them with empty strings (`""`)**.

Here's how it works in `utils/formatter.py`:

### Key Mechanism:
In the `to_csv()` function, when writing each record:
```python
writer.writerow({h: rec.get(h, "") for h in headers})
```
- For each header `h`, it uses `rec.get(h, "")`, which:
  - Returns the value if the key `h` exists in `rec`
  - Returns `""` (empty string) if the key is missing

### Evidence from Tests:
1. **`test_missing_fields_become_empty()`**:
   ```python
   records = [{"a": 1, "b": 2}, {"a": 3}]
   # Result: "a,b\r\n1,2\r\n3,\r\n"
   #         ↑ missing "b" in 2nd record → rendered as empty string
   ```

2. **`test_missing_keys_produce_empty_strings()`** (from `test_formatter_gen.py`):
   ```python
   records = [{"name": "Alice", "age": "30"}, {"name": "Bob"}]
   # Result: "name,age\r\nAlice,30\r\nBob,\r\n"
   #                                   ↑ empty for missing "age"
   ```

3. **`test_multiple_records_with_varying_keys()`**:
   ```python
   records = [
       {"a": "1", "b": "2"},
       {"b": "3", "c": "4"},  # missing "a" → empty string
       {"a": "5", "c": "6"},  # missing "b" → empty string
   ]
   # Result: "a,b,c\r\n1,2,\r\n,3,4\r\n5,,6\r\n"
   #                   ↑   ↑  ↑      ↑  ↑
   #                (a missing) (b missing) (b missing)
   ```

### Important Notes:
- All headers are collected from **all records** (first-seen order), so missing fields in *some* records are still 
covered in the CSV structure.
- The CSV writer (`csv.DictWriter`) is configured with `extrasaction="ignore"` to avoid errors from extra keys in 
individual records (though this doesn't affect missing keys).
- This behavior is **consistent and intentional** – missing fields → empty cells in the CSV output.

So yes, **missing fields become empty strings**, ensuring every row has the same number of columns.

---
## Chapter 6 — Grep

**Goal:** find files by what's *inside* them, not just their name.

Chapters 4–5 rank files by path relevance. That works well when the
query mentions a filename — "how does the *formatter* work?" — but fails
when the relevant file has a generic name.

Consider: *"Where does the code raise a TypeError?"*
No filename contains "typeerror". Fuzzy scoring gives every file a near-zero
path score. But `validator.py` *contains* `raise TypeError(...)` — and grep
finds it instantly.

Grep is the third retrieval strategy. It searches file content for
patterns derived from the query, counts matches per file, and uses that
count as an additional relevance signal on top of the fuzzy path score.

**You will:**
- Write `grep_repo()` — search file contents, return matches with excerpts
- Write `query_to_patterns()` — turn query terms into search patterns
- Write `grep_rank()` — combine grep hit count with fuzzy path score
- See grep surface `validator.py` for a query that fuzzy scoring misses entirely

### 6.1 `grep_repo()` — search file contents

For each file, we search for a compiled regex pattern and collect:
- the number of matching lines (used for ranking)
- up to `context_lines` surrounding each match (used in the prompt as an excerpt)

Returning excerpts rather than full file content is a key budget trick:
if grep finds 2 matching lines in a 300-line file, we can show just those
2 lines plus context instead of loading all 300 lines HOT.

In [146]:
def grep_repo(
    pattern:       str,
    repo_root:     str   = REPO_ROOT,
    extensions:    set   = CODE_EXTENSIONS,
    context_lines: int   = 2,
    max_matches:   int   = 5,
) -> list[dict]:
    """
    Search every matching file under *repo_root* for *pattern* (regex).

    Returns a list of dicts — one per file that had at least one match:
        {
          "path":      str,         # relative to repo_root
          "hit_count": int,         # number of matching lines
          "excerpt":   str,         # up to max_matches hits with context
          "bytes":     int,
          "hot":       False,       # budget_load() will flip this
          "score":     0.0,         # grep_rank() will fill this
        }
    Sorted by hit_count descending.
    """
    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error:
        regex = re.compile(re.escape(pattern), re.IGNORECASE)

    results = []
    root    = Path(repo_root)

    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".")]
        for fname in filenames:
            full = Path(dirpath) / fname
            if full.suffix.lower() not in extensions:
                continue
            try:
                lines = full.read_text(errors="ignore").splitlines()
            except OSError:
                continue

            # Find matching line indices
            hit_indices = [i for i, ln in enumerate(lines) if regex.search(ln)]
            if not hit_indices:
                continue

            # Build excerpt: up to max_matches hits with context
            excerpts, seen_lines = [], set()
            for hit_i in hit_indices[:max_matches]:
                start = max(0, hit_i - context_lines)
                end   = min(len(lines), hit_i + context_lines + 1)
                block = []
                for ln_i in range(start, end):
                    if ln_i not in seen_lines:
                        prefix = "→ " if ln_i == hit_i else "  "
                        block.append(f"{prefix}{ln_i+1:4d} │ {lines[ln_i]}")
                        seen_lines.add(ln_i)
                excerpts.append("\n".join(block))

            excerpt_text = "\n\n".join(excerpts)

            results.append({
                "path":      str(full.relative_to(root)),
                "hit_count": len(hit_indices),
                "excerpt":   excerpt_text,
                "bytes":     full.stat().st_size,
                "hot":       False,
                "score":     0.0,
            })

    return sorted(results, key=lambda x: x["hit_count"], reverse=True)


# ── Demo: search for TypeError ───────────────────────────────────────────────
hits = grep_repo("TypeError", repo_root="sample_project")
for h in hits:
    print(f"{h['hit_count']} hit(s)  {h['path']}")
    print(h["excerpt"])
    print()

3 hit(s)  utils/validator.py
     9 │     field names to expected types: "string", "number", "boolean").
    10 │ 
→   11 │     Raises TypeError on the first violation found.
    12 │     """
    13 │     schema: dict[str, str] = json.loads(schema_raw)

    17 │         for field, expected_type_name in schema.items():
    18 │             if field not in record:
→   19 │                 raise TypeError(
    20 │                     f"Record {i}: missing required field '{field}'"
    21 │                 )

    22 │             expected = type_map.get(expected_type_name)
    23 │             if expected and not isinstance(record[field], expected):
→   24 │                 raise TypeError(
    25 │                     f"Record {i}: field '{field}' expected {expected_type_name}, "
    26 │                     f"got {type(record[field]).__name__}"



### 6.2 `query_to_patterns()` — derive search patterns from a query

We turn the query terms into a single alternation regex:
`formatter|missing|fields`

This is a liberal match — any file mentioning *any* term is a candidate.
We'll use hit count to separate strong matches from weak ones.

In [147]:
def query_to_patterns(query: str) -> str:
    """
    Convert a natural-language query into a single regex pattern
    suitable for grep_repo().

    "Where does the code raise a TypeError?"
    → "typeerror|raise|code"   (terms ≥ 4 chars, lowercased, joined with |)

    Returns an empty string if no usable terms are found.
    """
    terms = [t for t in tokenize_query(query) if len(t) >= 4]
    if not terms:
        return ""
    return "|".join(re.escape(t) for t in terms)


# Check a few examples
for q in [
    "Where does the code raise a TypeError?",
    "How does the formatter handle missing fields?",
    "What is the CSV delimiter default value?",
]:
    print(f"  {q!r}\n  → {query_to_patterns(q)!r}\n")

  'Where does the code raise a TypeError?'
  → 'code|raise|typeerror'

  'How does the formatter handle missing fields?'
  → 'formatter|handle|missing|fields'

  'What is the CSV delimiter default value?'
  → 'delimiter|default|value'



### 6.3 `grep_rank()` — combine grep hits with fuzzy path score

A file can score high two ways:
- Many grep hits → strong content signal
- High fuzzy path score → strong name signal

`grep_rank()` normalises both signals to [0, 1] and adds them,
so a file with a relevant name *and* relevant content beats one with only one signal.

In [148]:
def grep_rank(
    query:     str,
    repo_root: str = REPO_ROOT,
) -> list[dict]:
    """
    Full grep-based retrieval pipeline for *query*:

    1. Derive a regex pattern from the query terms
    2. Grep every source file for that pattern
    3. Score each hit file with both grep hit_count and fuzzy path score
    4. Normalise both signals to [0, 1] and sum them
    5. Return sorted by combined score descending

    Files with zero grep hits are excluded entirely.
    """
    pattern = query_to_patterns(query)
    if not pattern:
        return []

    hits = grep_repo(pattern, repo_root=repo_root)
    if not hits:
        return []

    terms         = tokenize_query(query)
    max_hits      = max(h["hit_count"] for h in hits) or 1
    max_path_score = max(score_file(h, terms) for h in hits) or 1

    ranked = []
    for h in hits:
        norm_grep = h["hit_count"]   / max_hits
        norm_path = score_file(h, terms) / max_path_score
        combined  = round(norm_grep + norm_path, 4)
        ranked.append({**h, "score": combined})

    return sorted(ranked, key=lambda x: x["score"], reverse=True)


# ── Demo: the query fuzzy scoring fails on ───────────────────────────────────
query  = "Where does the code raise a TypeError?"
ranked = grep_rank(query, repo_root="sample_project")

tbl = Table(title=f"grep_rank(): {query!r}", header_style="bold cyan")
tbl.add_column("Score",    justify="right")
tbl.add_column("Hits",     justify="right")
tbl.add_column("File")
tbl.add_column("Excerpt (first match)", max_width=55, no_wrap=False)

for f in ranked:
    first_excerpt = f["excerpt"].splitlines()[0] if f["excerpt"] else ""
    tbl.add_row(
        f"[green]{f['score']}[/green]",
        str(f["hit_count"]),
        f["path"],
        first_excerpt,
    )
console.print(tbl)

# Compare: what fuzzy scoring alone would have returned
fuzzy_only = rank_files(glob_files("*.py", repo_root="sample_project"), query)
console.print("\n[bold]Fuzzy path scores for same query (for comparison):[/bold]")
for f in fuzzy_only:
    console.print(f"  {f['score']:.4f}  {f['path']}")

                      grep_rank(): 'Where does the code raise a TypeError?'                       
┏━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  Score ┃ Hits ┃ File                 ┃ Excerpt (first match)                                   ┃
┡━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    2.0 │    4 │ utils/parser.py      │     11 │     - a list of scalars   → each scalar        │
│        │      │                      │ wrapped as {"value": scalar}                            │
│ 1.4283 │    4 │ tests/test_parser.py │     18 │         assert result == [{"value": 1},        │
│        │      │                      │ {"value": 2}, {"value": 3}]                             │
│ 1.3561 │    3 │ utils/validator.py   │      9 │     field names to expected types: "string",   │
│        │      │                      │ "number", "boolean").                                   │
└────────┴──────┴──────────────────────┴─────────────────────────────────────────────────────────┘

Fuzzy path scores for same query (for comparison):

1.3083  utils/parser.py

1.2325  utils/formatter.py

0.7929  utils/validator.py

0.5604  tests/test_parser.py

0.5491  tests/test_formatter.py

0.4757  tests/test_formatter_gen.py

0.4444  main.py

### 6.4 Try it — grep-guided query with excerpt injection

One more trick: instead of loading the *full* file HOT, we can inject
just the grep **excerpt** for files that matched well but are expensive
to load in full. This stretches the budget further.

For this demo we load full content (the sample files are small), but
the `excerpt` field is there and Chapter 7 will start using it during compaction.

In [149]:
query = "Where does the code raise a TypeError?"
repo  = "sample_project"

# Grep-ranked candidates (includes excerpt, excludes files with zero hits)
candidates = grep_rank(query, repo_root=repo)

# Fall back to fuzzy glob for files with zero grep hits
all_paths    = {f["path"] for f in candidates}
glob_extras  = [f for f in rank_files(glob_files("*.py", repo_root=repo), query)
                if f["path"] not in all_paths]
candidates   = candidates + glob_extras   # grep hits first, fuzzy extras after

# Budget-aware JIT load
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

console.rule("[bold]Chapter 6 — grep-ranked before LLM call[/bold]")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "grep + fuzzy + JIT",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

console.rule("[bold]After call[/bold]")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "grep + fuzzy + JIT",
    prompt_size = prompt_size,
)
console.print(f"\n[bold green]Response:[/bold green]\n{reply}")

───────────────────────────────────── Chapter 6 — grep-ranked before LLM call ─────────────────────────────────────

╭──────── Pocket Agent  ·  Where does the code raise a TypeError? ─────────╮
│ Token Budget      ██████████████░░░░░░░░░░░░░░░░░░  1,913 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/parser.py                                │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/validator.py                             │
│                    HOT    utils/formatter.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    main.py                                        │
│                                                                          │
│ Strategy          grep + fuzzy + JIT                                     │
│ Prompt size       1,913 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

╭──────── Pocket Agent  ·  Where does the code raise a TypeError? ─────────╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  1,984 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/parser.py                                │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/validator.py                             │
│                    HOT    utils/formatter.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    main.py                                        │
│                                                                          │
│ Strategy          manifest                                               │
│ Prompt size       1,984 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── After call ────────────────────────────────────────────────────

╭──────── Pocket Agent  ·  Where does the code raise a TypeError? ─────────╮
│ Token Budget      ██████████████████░░░░░░░░░░░░░░  2,384 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/parser.py                                │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/validator.py                             │
│                    HOT    utils/formatter.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    main.py                                        │
│                                                                          │
│ Strategy          grep + fuzzy + JIT                                     │
│ Prompt size       1,913 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

Response:
The code raises a `TypeError` in **`utils/validator.py`**, specifically in the `validate_schema` function.

Here’s where and why:

```python
# In utils/validator.py, validate_schema()
expected = type_map.get(expected_type_name)
if expected and not isinstance(record, expected):
    raise TypeError(
        f"Record {i}: field '{field}' expected {expected_type_name}, "
        f"got {type(record).__name__}"
    )
```

### When does it raise `TypeError`?
1. **If a required field is missing** in a record:
   ```python
   if field not in record:
       raise TypeError(f"Record {i}: missing required field '{field}'")
   ```
2. **If a field's value is not of the expected type**:
   For example, if the schema says `"age": "number"` but the record contains `"age": "thirty"` (a string), then:
   ```python
   isinstance("thirty", (int, float)) → False
   → raises TypeError
   ```

### Example trigger:
Given:
- Schema: `'{"name": "string", "age": "number"}'`
- Record: `{"name": "Alice", "age": "30"}` (string `"30"` instead of number `30`)

Then `validate_schema([{"name": "Alice", "age": "30"}], schema)` raises:
```
TypeError: Record 0: field 'age' expected number, got str
```

### ✅ Summary:
- **Location**: `utils/validator.py`, inside `validate_schema()`.
- **Trigger**: Missing field or type mismatch.
- **Not raised anywhere else** in the provided files — other files use `ValueError` for parsing/formatting errors.

Let me know if you'd like to add a test to verify the `TypeError` behavior!

---
## Chapter 7 — Microcompaction

**Goal:** survive long sessions without hitting the context ceiling.

After several turns the conversation grows: loaded file contents, prior replies,
system prompts. Eventually the budget fills up and the model starts forgetting
the beginning of the conversation.

Microcompaction solves this with a simple rule:

> When the running token count exceeds a **compaction threshold**,
> take the coldest HOT files, summarise each one in a single LLM call,
> replace the full content with the summary, and mark them COLD.

The summary is much smaller than the original — typically 10–15 % of the
original token count. Budget is freed; the session continues.

```
Before compaction            After compaction
─────────────────            ────────────────
HOT  utils/parser.py  420t   COLD utils/parser.py  [summary: 45t]
HOT  utils/formatter  380t   HOT  utils/formatter  380t   ← kept
HOT  utils/validator  290t   HOT  utils/validator  290t   ← kept
─────────────────            ────────────────
total: 1090t                 total: 715t   (saved 375t)
```

**You will:**
- Write `eviction_candidates()` — pick which HOT files to evict
- Write `summarise_file()` — compress one file to a short summary via Ollama
- Write `compact()` — orchestrate eviction + summarisation, return updated file list
- Simulate a session running over budget and watch compaction kick in

### 7.1 `eviction_candidates()` — which HOT files to evict first

We evict the files that are least likely to be needed again.
The heuristic: **lowest score first** (from Chapter 5's fuzzy scoring).
Files with no score get evicted before files that matched the current query.

We also always keep at least one HOT file — evicting everything defeats the purpose.

In [150]:
COMPACT_THRESHOLD = 0.80   # trigger compaction when prompt > 80 % of budget
EVICT_TARGET      = 0.55   # compact down to 55 % of budget


def eviction_candidates(
    files:    list[dict],
    query:    str,
    n_keep:   int = 1,
) -> tuple[list[dict], list[dict]]:
    """
    Split *files* into (evict, keep).

    HOT files are sorted by their relevance score (ascending — lowest first).
    We evict all but the top *n_keep* HOT files.
    COLD files are never touched here.

    Returns (to_evict, to_keep) — both lists contain the original dicts.
    """
    hot  = [f for f in files if f.get("hot")]
    cold = [f for f in files if not f.get("hot")]

    if len(hot) <= n_keep:
        return [], files   # nothing to evict

    terms  = tokenize_query(query)
    # Sort HOT by score ascending (lowest relevance evicted first)
    scored = sorted(hot, key=lambda f: f.get("score", score_file(f, terms)))

    to_evict = scored[:-n_keep]    # all except the top n_keep
    to_keep  = scored[-n_keep:] + cold

    return to_evict, to_keep


# ── Quick check ──────────────────────────────────────────────────────────────
_demo_files = [
    {"path": "utils/parser.py",    "hot": True,  "score": 0.8,  "tokens": 420},
    {"path": "utils/formatter.py", "hot": True,  "score": 2.1,  "tokens": 380},
    {"path": "utils/validator.py", "hot": True,  "score": 0.3,  "tokens": 290},
    {"path": "main.py",            "hot": False, "score": 0.1,  "tokens": 120},
]
evict, keep = eviction_candidates(_demo_files, "how does the formatter work?")
print("Evict:", [f["path"] for f in evict])
print("Keep: ", [f["path"] for f in keep])

Evict: ['utils/validator.py', 'utils/parser.py']
Keep:  ['utils/formatter.py', 'main.py']


### 7.2 `summarise_file()` — compress one file with the LLM

We ask the model for a terse technical summary in plain prose — no code blocks,
no bullet points. The goal is a 3–5 sentence description that preserves
the key exported names and their purpose, so later queries can still
reference this file even though its full content is no longer in the prompt.

In [151]:
def summarise_file(file_dict: dict) -> dict:
    """
    Ask the LLM to compress *file_dict["content"]* to a short summary.

    Returns a new dict with:
      - "content"  replaced by the summary text
      - "tokens"   updated to the summary's token count
      - "hot"      set to False  (evicted from active prompt)
      - "summary"  set to True   (flag so later code knows this is compressed)
    """
    content = file_dict.get("content", "")
    if not content.strip():
        return {**file_dict, "hot": False, "summary": True}

    prompt = (
        f"Summarise the following source file in 3–5 sentences of plain prose. "
        f"Name the key functions/classes and what they do. "
        f"No code blocks, no bullet points.\n\n"
        f"FILE: {file_dict['path']}\n\n{content}"
    )
    summary_text, _ = chat([{"role": "user", "content": prompt}])

    return {
        **file_dict,
        "content": f"[SUMMARY of {file_dict['path']}]\n{summary_text}",
        "tokens":  count_tokens(summary_text),
        "hot":     False,
        "summary": True,
    }


# ── Demo: summarise parser.py ────────────────────────────────────────────────
_parser_meta   = glob_files("parser.py", repo_root="sample_project")[0]
_parser_loaded = jit_read(_parser_meta, repo_root="sample_project")

console.print(f"[bold]Before:[/bold] {_parser_loaded['tokens']} tokens")
_parser_summary = summarise_file(_parser_loaded)
console.print(f"[bold]After :[/bold] {_parser_summary['tokens']} tokens  "
              f"({_parser_summary['tokens']/_parser_loaded['tokens']*100:.0f}% of original)\n")
console.print(_parser_summary["content"])

Before: 303 tokens

After : 173 tokens  (57% of original)

[SUMMARY of utils/parser.py]
The `parse_json` function takes a JSON string and converts it into a list of flat dictionaries, handling cases 
where the input is a list of objects, a single object, or a list of scalar values—wrapping scalars as `{"value": 
scalar}` and raising a `ValueError` for malformed or unsupported JSON. It relies on the helper function `_flatten`,
which recursively processes nested dictionaries to produce flat dictionaries with dot-separated keys, such as 
converting `{"a": {"b": 1}}` into `{"a.b": 1}`. If the input JSON is not a list or dictionary, `parse_json` raises 
a `ValueError`. Both functions ensure robust parsing and normalization of diverse JSON structures into a consistent
flat format.

### 7.3 `compact()` — orchestrate the full compaction pass

`compact()` is called by the agent whenever the token count crosses
`COMPACT_THRESHOLD`. It loops through eviction candidates, summarises
each one, and returns the updated file list plus a log of what was freed.

In [152]:
def compact(
    files:      list[dict],
    query:      str,
    token_used: int,
) -> tuple[list[dict], int, list[str]]:
    """
    If *token_used* exceeds COMPACT_THRESHOLD × TOKEN_BUDGET, evict and
    summarise the least-relevant HOT files until usage drops below
    EVICT_TARGET × TOKEN_BUDGET.

    Returns:
        (updated_files, new_token_used, log_lines)
    where log_lines is a human-readable list of what was compacted.
    """
    threshold = int(TOKEN_BUDGET * COMPACT_THRESHOLD)
    target    = int(TOKEN_BUDGET * EVICT_TARGET)

    if token_used <= threshold:
        return files, token_used, []   # nothing to do

    log     = [f"Compaction triggered: {token_used:,} / {TOKEN_BUDGET:,} tokens "
               f"({token_used/TOKEN_BUDGET*100:.0f}%)"]
    updated = list(files)
    used    = token_used

    while used > target:
        to_evict, _ = eviction_candidates(updated, query, n_keep=1)
        if not to_evict:
            break

        # Evict one file at a time
        victim = to_evict[0]
        before = victim.get("tokens", 0)

        summarised = summarise_file(victim)
        after      = summarised["tokens"]
        saved      = before - after

        # Replace the victim in the list
        updated = [summarised if f["path"] == victim["path"] else f
                   for f in updated]
        used   -= saved
        log.append(f"  compacted {victim['path']}: {before}t → {after}t  (saved {saved}t)")

    log.append(f"After compaction: {used:,} / {TOKEN_BUDGET:,} tokens "
               f"({used/TOKEN_BUDGET*100:.0f}%)")
    return updated, used, log

### 7.4 Simulation — watch compaction fire

We simulate a session that loads all sample files and artificially inflates
the token count past the compaction threshold, then watch `compact()` log
what it evicts and how much budget it recovers.

In [153]:
query = "How does the formatter handle missing fields?"
repo  = "sample_project"

# Load all Python files HOT
all_files   = glob_files("*.py", repo_root=repo)
loaded      = [jit_read(f, repo_root=repo) for f in all_files]
# Give them fuzzy scores so eviction ordering works
terms       = tokenize_query(query)
loaded      = [{**f, "score": score_file(f, terms)} for f in loaded]
token_used  = sum(f["tokens"] for f in loaded)

# Artificially inflate to simulate a long session
# (pad to 85 % of budget so compaction fires)
pad_tokens  = int(TOKEN_BUDGET * 0.85) - token_used
token_used += max(pad_tokens, 0)

console.rule("[bold]Before compaction[/bold]")
show_panel(
    query        = query,
    token_used   = token_used,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded],
    strategy     = "compact",
    prompt_size  = token_used,
)

# Run compaction
updated, new_used, log = compact(loaded, query, token_used)

console.rule("[bold]Compaction log[/bold]")
for line in log:
    console.print(f"  {line}")

console.rule("[bold]After compaction[/bold]")
show_panel(
    query        = query,
    token_used   = new_used,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in updated],
    strategy     = "compact",
    prompt_size  = new_used,
)

──────────────────────────────────────────────── Before compaction ────────────────────────────────────────────────

╭───── Pocket Agent  ·  How does the formatter handle missing fields? ─────╮
│ Token Budget      ███████████████████████████░░░░░  3,481 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    main.py                                        │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/formatter.py                             │
│                    HOT    utils/parser.py                                │
│                    HOT    utils/validator.py                             │
│                                                                          │
│ Strategy          compact                                                │
│ Prompt size       3,481 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────────────── Compaction log ──────────────────────────────────────────────────

Compaction triggered: 3,481 / 4,096 tokens (85%)

compacted tests/test_parser.py: 279t → 172t  (saved 107t)

...

compacted utils/parser.py: 303t → 197t  (saved 106t)

compacted main.py: 173t → 131t  (saved 42t)

After compaction: 2,765 / 4,096 tokens (68%)

──────────────────────────────────────────────── After compaction ─────────────────────────────────────────────────

╭───── Pocket Agent  ·  How does the formatter handle missing fields? ─────╮
│ Token Budget      █████████████████████░░░░░░░░░░░  2,765 / 4,096 tokens │
│                                                                          │
│ Retrieved          COLD   main.py                                        │
│                    COLD   tests/test_formatter.py                        │
│                    COLD   tests/test_formatter_gen.py                    │
│                    COLD   tests/test_parser.py                           │
│                    HOT    utils/formatter.py                             │
│                    COLD   utils/parser.py                                │
│                    COLD   utils/validator.py                             │
│                                                                          │
│ Strategy          compact                                                │
│ Prompt size       2,765 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

---
## Chapter 8 — Semantic RAG

**Goal:** retrieve files by meaning, not just keywords or filenames.

Grep and fuzzy scoring both rely on surface-level text overlap.
They miss cases like: *"where is the nesting logic handled?"* — the relevant
function is `_flatten` in `parser.py`, but neither the query nor the filename
contains the word "flatten".

Semantic retrieval fixes this by:
1. **Embedding** each file (or chunk) into a vector using `nomic-embed-text`
2. **Embedding** the query into the same vector space at runtime
3. **Ranking** files by cosine similarity — close vectors = similar meaning

The embeddings are computed once and cached in memory. On subsequent calls
only the (cheap) query embedding is recomputed.

**You will:**
- Write `embed()` — call Ollama's embedding endpoint
- Write `build_index()` — embed all files and store vectors in memory
- Write `semantic_retrieve()` — embed the query and return ranked files
- Compare semantic vs grep for a query that grep misses

### 8.1 Dependencies

Chapter 8 needs `numpy` for the cosine similarity calculation.
Make sure `nomic-embed-text` is pulled — this is the `OLLAMA_EMBED` model
set in the config cell at the top.

In [154]:
import subprocess, sys

_CH8_DEPS = ["numpy"]
for _pkg in _CH8_DEPS:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", _pkg],
        stdout=subprocess.DEVNULL,
    )

import numpy as np
print("Chapter 8 dependencies ready:", _CH8_DEPS)

# Confirm the embed model is available
ok, models = ping_ollama()
if ok and any(OLLAMA_EMBED in m for m in models):
    print(f"Embed model '{OLLAMA_EMBED}' is available.")
else:
    print(f"WARNING: embed model '{OLLAMA_EMBED}' not found.")
    print(f"Run: ollama pull {OLLAMA_EMBED}")

Chapter 8 dependencies ready: ['numpy']
Embed model 'nomic-embed-text' is available.


### 8.2 `embed()` — call Ollama's embedding endpoint

Ollama exposes `/api/embed` (or `/api/embeddings` in older versions).
It accepts a model name and a string, returns a float vector.
We normalise the vector to unit length immediately so cosine similarity
later is just a dot product.

In [155]:
def embed(text: str, model: str = OLLAMA_EMBED) -> np.ndarray:
    """
    Return a unit-normalised embedding vector for *text*.

    Uses Ollama's /api/embed endpoint (introduced in Ollama 0.1.26).
    Falls back to /api/embeddings for older installations.
    """
    payload = {"model": model, "input": text}
    try:
        r = requests.post(f"{OLLAMA_BASE_URL}/api/embed",
                          json=payload, timeout=60)
        r.raise_for_status()
        vec = r.json()["embeddings"][0]
    except Exception:
        # Fallback for older Ollama versions
        payload_old = {"model": model, "prompt": text}
        r = requests.post(f"{OLLAMA_BASE_URL}/api/embeddings",
                          json=payload_old, timeout=60)
        r.raise_for_status()
        vec = r.json()["embedding"]

    arr  = np.array(vec, dtype=np.float32)
    norm = np.linalg.norm(arr)
    return arr / norm if norm > 0 else arr


# Quick smoke test
_v = embed("hello world")
print(f"Vector dimension : {len(_v)}")
print(f"Vector norm      : {np.linalg.norm(_v):.6f}  (should be ~1.0)")

Vector dimension : 768
Vector norm      : 1.000000  (should be ~1.0)


### 8.3 `build_index()` — embed every file, cache in memory

We embed each file's content and store the vector alongside the file metadata.
This is the expensive step — one embedding call per file.
After the first call the index lives in the `_EMBED_INDEX` dict and is reused.

In [156]:
# In-memory index: repo_root → list of {"path", "vector", "bytes", "hot"}
_EMBED_INDEX: dict[str, list[dict]] = {}


def build_index(
    repo_root:  str = REPO_ROOT,
    extensions: set = CODE_EXTENSIONS,
    force:      bool = False,
) -> list[dict]:
    """
    Embed every source file under *repo_root* and cache the result.

    Returns the index (list of dicts with a "vector" key).
    On subsequent calls the cached index is returned unless *force=True*.
    """
    if repo_root in _EMBED_INDEX and not force:
        return _EMBED_INDEX[repo_root]

    files = glob_files("*", repo_root=repo_root)
    files = [f for f in files
             if Path(f["path"]).suffix.lower() in extensions]

    index = []
    for f in files:
        full_path = Path(repo_root) / f["path"]
        try:
            content = full_path.read_text(errors="ignore")
        except OSError:
            continue
        # Truncate very long files before embedding to avoid timeouts
        snippet = content[:4000]
        vec = embed(snippet)
        index.append({**f, "vector": vec, "content": content,
                      "tokens": count_tokens(content)})
        console.print(f"  embedded  {f['path']}")

    _EMBED_INDEX[repo_root] = index
    return index


console.rule("[bold]Building semantic index for sample_project[/bold]")
idx = build_index(repo_root="sample_project")
console.print(f"\nIndex contains {len(idx)} files.")

─────────────────────────────────── Building semantic index for sample_project ────────────────────────────────────

embedded  main.py

embedded  tests/test_formatter.py

embedded  tests/test_formatter_gen.py

embedded  tests/test_parser.py

embedded  utils/formatter.py

embedded  utils/parser.py

embedded  utils/validator.py

Index contains 7 files.

### 8.4 `semantic_retrieve()` — rank by cosine similarity

Cosine similarity between unit vectors is just a dot product.
We embed the query, dot it against every file vector, and sort descending.
The result dict shape matches what `budget_load()` and `show_panel()` expect.

In [157]:
def semantic_retrieve(
    query:     str,
    repo_root: str = REPO_ROOT,
    top_k:     int = 5,
) -> list[dict]:
    """
    Return the top-*k* files from the semantic index, ranked by cosine
    similarity to *query*.

    Each returned dict has a "score" key (cosine similarity, 0–1).
    """
    index     = build_index(repo_root=repo_root)
    query_vec = embed(query)

    scored = []
    for entry in index:
        sim = float(np.dot(query_vec, entry["vector"]))
        scored.append({
            "path":    entry["path"],
            "bytes":   entry["bytes"],
            "tokens":  entry["tokens"],
            "content": entry["content"],
            "hot":     False,
            "score":   round(sim, 4),
        })

    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]


# ── Demo: the query grep struggles with ──────────────────────────────────────
query = "Where is the nesting logic handled?"

sem_results  = semantic_retrieve(query, repo_root="sample_project")
grep_results = grep_rank(query, repo_root="sample_project")

tbl = Table(title="Semantic vs Grep", header_style="bold cyan")
tbl.add_column("Method")
tbl.add_column("Rank", justify="right")
tbl.add_column("Score", justify="right")
tbl.add_column("File")

for i, f in enumerate(sem_results, 1):
    tbl.add_row("semantic", str(i), f"[green]{f['score']}[/green]", f["path"])

for i, f in enumerate(grep_results, 1):
    tbl.add_row("grep",     str(i), f"[yellow]{f['score']}[/yellow]", f["path"])

console.print(tbl)

                 Semantic vs Grep                  
┏━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Method   ┃ Rank ┃  Score ┃ File                 ┃
┡━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ semantic │    1 │ 0.5435 │ main.py              │
│ semantic │    2 │ 0.5421 │ utils/validator.py   │
│ semantic │    3 │ 0.5225 │ tests/test_parser.py │
│ semantic │    4 │ 0.5159 │ utils/formatter.py   │
│ semantic │    5 │ 0.5073 │ utils/parser.py      │
└──────────┴──────┴────────┴──────────────────────┘

### 8.5 Try it end-to-end

Ask the nesting query. Semantic retrieval finds `parser.py` at the top
because `_flatten` is semantically close to "nesting logic",
even though neither word appears in the other.

In [158]:
query = "Where is the nesting logic handled?"
repo  = "sample_project"

candidates   = semantic_retrieve(query, repo_root=repo)
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

console.rule("[bold]Chapter 8 — semantic retrieval before LLM call[/bold]")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "semantic RAG",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

console.rule("[bold]After call[/bold]")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "semantic RAG",
    prompt_size = prompt_size,
)
console.print(f"\n[bold green]Response:[/bold green]\n{reply}")

───────────────────────────────── Chapter 8 — semantic retrieval before LLM call ──────────────────────────────────

╭────────── Pocket Agent  ·  Where is the nesting logic handled? ──────────╮
│ Token Budget      █████████░░░░░░░░░░░░░░░░░░░░░░░  1,247 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    main.py                                        │
│                    HOT    utils/validator.py                             │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/formatter.py                             │
│                    HOT    utils/parser.py                                │
│                                                                          │
│ Strategy          semantic RAG                                           │
│ Prompt size       1,247 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

╭────────── Pocket Agent  ·  Where is the nesting logic handled? ──────────╮
│ Token Budget      ██████████░░░░░░░░░░░░░░░░░░░░░░  1,296 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    main.py                                        │
│                    HOT    utils/validator.py                             │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/formatter.py                             │
│                    HOT    utils/parser.py                                │
│                                                                          │
│ Strategy          manifest                                               │
│ Prompt size       1,296 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── After call ────────────────────────────────────────────────────

╭────────── Pocket Agent  ·  Where is the nesting logic handled? ──────────╮
│ Token Budget      ████████████░░░░░░░░░░░░░░░░░░░░  1,583 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    main.py                                        │
│                    HOT    utils/validator.py                             │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/formatter.py                             │
│                    HOT    utils/parser.py                                │
│                                                                          │
│ Strategy          semantic RAG                                           │
│ Prompt size       1,247 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

Response:
The **nesting logic** is handled in the `_flatten` function in `utils/parser.py`.

### Specifically:
```python
def _flatten(obj: dict, prefix: str = "", sep: str = ".") -> dict:
    """Recursively flatten nested dicts: {"a": {"b": 1}} → {"a.b": 1}."""
    result = {}
    for key, value in obj.items():
        full_key = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            result.update(_flatten(value, full_key, sep))  # ← recursive call
        else:
            result = value
    return result
```

### Where it's used:
- Inside `parse_json`, when processing each item:
  - If the item is a `dict`, it's passed to `_flatten`.
  - If the item is a scalar (e.g., number, string) in a list, it's wrapped as `{"value": scalar}` — **no flattening
needed here**.
- This ensures all nested objects are recursively flattened into flat dicts with dot-notation keys.

### Example:
Input JSON:
```json
{
  "a": {
    "b": {
      "c": 42
    }
  }
}
```
→ After `_flatten`: `{"a.b.c": 42}`

So the **core nesting/flatting logic is in `_flatten`**, and it's **invoked by `parse_json`** for any dictionary 
input.

---
## Chapter 9 — Full Pipeline

**Goal:** one function call, `run(query, repo)`, does everything.

Every chapter so far introduced one tool. Chapter 9 wires them together:

```
query
  │
  ├─ manifest loaded (always)
  │
  ├─ strategy picker
  │     ├─ "semantic"  → semantic_retrieve()
  │     ├─ "grep"      → grep_rank()
  │     └─ "fuzzy"     → rank_files(glob_files())
  │
  ├─ budget_load()   — JIT read, HOT/COLD split
  │
  ├─ compact()       — if > COMPACT_THRESHOLD
  │
  └─ ask_with_manifest()  → reply
```

The **strategy picker** is a small classifier that looks at the query:
- Contains a specific symbol or quoted string → grep (looking for code)
- Vague / conceptual phrasing → semantic (understanding meaning)
- Everything else → fuzzy (fast, good enough for named-file queries)

**You will:**
- Write `pick_strategy()` — classify the query
- Write `run()` — the full pipeline in one call
- Try three queries that each route to a different strategy

### 9.1 `pick_strategy()` — classify the query

A lightweight heuristic — no ML needed.
The three signals are mutually exclusive and checked in priority order.

In [159]:
# Words that suggest the user is asking about a concept, not looking for a symbol
_CONCEPTUAL_WORDS = {
    "why", "how", "explain", "what", "design", "architecture",
    "pattern", "approach", "concept", "idea", "strategy", "logic",
    "purpose", "reason", "difference", "relationship",
}

# Patterns that suggest the user is looking for a specific symbol or string
_GREP_SIGNALS = re.compile(
    r'(?:'
    r'"[^"]+"'              # quoted string
    r'|`[^`]+`'             # backtick symbol
    r'|raise\s+\w+'         # raise SomeError
    r'|def\s+\w+'           # def function_name
    r'|class\s+\w+'         # class ClassName
    r'|import\s+\w+'        # import something
    r'|\b[A-Z][a-zA-Z]+Error\b'  # ExceptionName
    r')'
)


def pick_strategy(query: str) -> str:
    """
    Classify *query* into one of three retrieval strategies:
      "grep"     — looking for a specific symbol / string in code
      "semantic" — conceptual / meaning-based question
      "fuzzy"    — everything else (filename-level search)
    """
    # Grep signal: backticks, quotes, error names, def/class/import
    if _GREP_SIGNALS.search(query):
        return "grep"

    # Semantic signal: conceptual vocabulary
    words = set(query.lower().split())
    if words & _CONCEPTUAL_WORDS:
        return "semantic"

    return "fuzzy"


# ── Check the three example queries ──────────────────────────────────────────
for q in [
    "Where does the code raise a `TypeError`?",
    "Why does the parser wrap scalars in a dict?",
    "How does the formatter handle missing fields?",
]:
    print(f"  {pick_strategy(q):8s}  {q}")

  grep      Where does the code raise a `TypeError`?
  semantic  Why does the parser wrap scalars in a dict?
  semantic  How does the formatter handle missing fields?


### 9.2 `run()` — the full pipeline

This is the function all later chapters call.
It returns a `RunResult` named tuple so callers can inspect
the files that were loaded, the strategy used, and the token counts.

In [160]:
from typing import NamedTuple


class RunResult(NamedTuple):
    reply:       str
    strategy:    str
    files:       list[dict]   # full list, HOT and COLD
    tokens_used: int
    compact_log: list[str]    # empty if compaction didn't fire


def run(
    query:     str,
    repo_root: str  = REPO_ROOT,
    strategy:  str  = "auto",   # "auto" | "fuzzy" | "grep" | "semantic"
    top_k:     int  = 8,
) -> RunResult:
    """
    Full retrieval + generation pipeline.

    Parameters
    ----------
    query     : the user's question
    repo_root : path to the repository to search
    strategy  : retrieval strategy; "auto" lets pick_strategy() decide
    top_k     : max candidates to consider before budget_load()
    """
    # ── 1. Manifest ─────────────────────────────────────────────────────────
    manifest     = load_manifest(repo_root)
    manifest_tok = manifest["tokens"]

    # ── 2. Strategy selection ────────────────────────────────────────────────
    strat = pick_strategy(query) if strategy == "auto" else strategy

    # ── 3. Retrieval ─────────────────────────────────────────────────────────
    if strat == "grep":
        candidates = grep_rank(query, repo_root=repo_root)
        # Append fuzzy extras for files with zero grep hits
        found_paths = {f["path"] for f in candidates}
        extras = [f for f in rank_files(glob_files("*.py", repo_root=repo_root), query)
                  if f["path"] not in found_paths]
        candidates = (candidates + extras)[:top_k]

    elif strat == "semantic":
        candidates = semantic_retrieve(query, repo_root=repo_root, top_k=top_k)

    else:   # fuzzy
        candidates = rank_files(glob_files("*.py", repo_root=repo_root), query)[:top_k]

    # ── 4. Budget-aware JIT load ─────────────────────────────────────────────
    loaded = budget_load(candidates, already_used=manifest_tok, repo_root=repo_root)

    # ── 5. Compaction (if needed) ─────────────────────────────────────────────
    hot_tok = sum(f.get("tokens", 0) for f in loaded if f["hot"])
    total   = manifest_tok + hot_tok + count_tokens(query)
    loaded, total, compact_log = compact(loaded, query, total)

    # ── 6. Show panel ─────────────────────────────────────────────────────────
    hot_files = [f for f in loaded if f["hot"]]
    show_panel(
        query        = query,
        token_used   = total,
        files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded],
        strategy     = strat,
        prompt_size  = total,
    )

    # ── 7. Generate reply ─────────────────────────────────────────────────────
    reply, tokens_used = ask_with_manifest(query, repo_root=repo_root, files=hot_files)

    return RunResult(
        reply        = reply,
        strategy     = strat,
        files        = loaded,
        tokens_used  = tokens_used,
        compact_log  = compact_log,
    )

### 9.3 Try all three strategies

Three queries, each routed automatically to a different strategy.
Watch the panel's **Strategy** field change on each run.

In [161]:
repo = "sample_project"

queries = [
    "Where does the code raise a `TypeError`?",           # → grep
    "Why does the parser wrap scalars in a dict?",        # → semantic
    "What does the formatter do with the delimiter?",     # → fuzzy
]

for q in queries:
    console.rule(f"[bold]query: {q}[/bold]")
    result = run(q, repo_root=repo)
    console.print(f"\n[bold]Strategy used:[/bold] {result.strategy}")
    console.print(f"[bold green]Response:[/bold green]\n{result.reply}\n")

───────────────────────────────── query: Where does the code raise a `TypeError`? ─────────────────────────────────

╭─────── Pocket Agent  ·  Where does the code raise a `TypeError`? ────────╮
│ Token Budget      ██████████████░░░░░░░░░░░░░░░░░░  1,914 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/parser.py                                │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/validator.py                             │
│                    HOT    utils/formatter.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    main.py                                        │
│                                                                          │
│ Strategy          grep                                                   │
│ Prompt size       1,914 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

╭─────── Pocket Agent  ·  Where does the code raise a `TypeError`? ────────╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  1,984 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    utils/parser.py                                │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/validator.py                             │
│                    HOT    utils/formatter.py                             │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    main.py                                        │
│                                                                          │
│ Strategy          manifest                                               │
│ Prompt size       1,984 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

Strategy used: grep

Response:
The code raises a `TypeError` in the **`utils/validator.py`** file, specifically in the `validate_schema` function.

Here's where and why:

### ✅ Location: `utils/validator.py`, inside `validate_schema`

The `TypeError` is raised **explicitly** in two places:

#### 1. **When a required field is missing**
```python
if field not in record:
    raise TypeError(
        f"Record {i}: missing required field '{field}'"
    )
```

#### 2. **When a field has the wrong type**
```python
if expected and not isinstance(record, expected):
    raise TypeError(
        f"Record {i}: field '{field}' expected {expected_type_name}, "
        f"got {type(record).__name__}"
    )
```

So a `TypeError` occurs **only if**:
- A record is missing a field defined in the schema (`field not in record`), or
- A field's value is not of the expected type (`isinstance(...)` fails).

> 🔍 Note: This is different from `ValueError`, which is raised in `utils/parser.py` for malformed JSON or 
unexpected top-level JSON types.

---

### Example when `TypeError` is raised:
Suppose this schema is used:
```json
{"name": "string", "age": "number"}
```
And the parsed records are:
```python
[{"name": "Alice"}]
```
→ `TypeError`: `"Record 0: missing required field 'age'"`

Or if:
```python
[{"name": "Alice", "age": "thirty"}]
```
→ `TypeError`: `"Record 0: field 'age' expected number, got str"`

---

✅ **Confirmed**: This matches the docstring of `validate_schema`: *“Raises TypeError on the first violation 
found.”*

...

────────────────────────────── query: What does the formatter do with the delimiter? ──────────────────────────────

╭──── Pocket Agent  ·  What does the formatter do with the delimiter? ─────╮
│ Token Budget      ██████████████░░░░░░░░░░░░░░░░░░  1,915 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    tests/test_formatter.py                        │
│                    HOT    utils/formatter.py                             │
│                    HOT    main.py                                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    utils/parser.py                                │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/validator.py                             │
│                                                                          │
│ Strategy          semantic                                               │
│ Prompt size       1,915 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

╭──── Pocket Agent  ·  What does the formatter do with the delimiter? ─────╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  1,986 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    tests/test_formatter.py                        │
│                    HOT    utils/formatter.py                             │
│                    HOT    main.py                                        │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    utils/parser.py                                │
│                    HOT    tests/test_parser.py                           │
│                    HOT    utils/validator.py                             │
│                                                                          │
│ Strategy          manifest                                               │
│ Prompt size       1,986 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

Strategy used: semantic

Response:
Looking at the code, the **formatter uses the `delimiter` parameter to configure the CSV writer**.

In `utils/formatter.py`, the `to_csv` function accepts a `delimiter` parameter (defaulting to `","`) and passes it 
to `csv.DictWriter`:

```python
writer = csv.DictWriter(buf, fieldnames=headers,
                        delimiter=delimiter, extrasaction="ignore")
```

Here's what happens with the delimiter:

1. **It's used by the CSV writer** to determine how to separate columns in the output
2. **Default is comma (`,`)** - when no delimiter is specified
3. **Custom delimiters work** - as shown in the test `test_custom_delimiter` which uses `"\t"` (tab)
4. **The delimiter affects the entire CSV output** - all field separators in the header row and data rows use this 
character

For example:
- Default comma: `"name,age\r\nAlice,30\r\n"`
- Tab delimiter: `"name\tscore\r\nAlice\t95\r\n"`

The `csv.DictWriter` from Python's standard library handles the delimiter transparently - it automatically uses the
specified delimiter character instead of commas when writing the CSV content.

So the formatter doesn't manually process the delimiter - it delegates this responsibility to Python's built-in CSV
module, which correctly handles the delimiter for all fields in the output.

---
## Chapter 10 — Write + Diff

**Goal:** give the agent the ability to modify files on disk.

Up to now the agent only reads. Chapter 10 adds the write side:
the agent proposes a change, we show a coloured unified diff,
and only write if confirmed (or if `dry_run=False` is set explicitly).

Two principles guide this chapter:

1. **Show before you write.** The diff is always printed before any file is touched.
2. **Patch, don't replace.** The agent receives the original file content
   and returns the complete new content. We generate the diff from those two
   strings — no fragile line-number arithmetic.

**You will:**
- Write `make_diff()` — unified diff between two strings, with colour
- Write `apply_patch()` — ask the LLM to rewrite a file given an instruction
- Write `write_file()` — diff + write, with dry-run support
- Try it: ask the agent to add a docstring to a function

### 10.1 `make_diff()` — coloured unified diff

Python's `difflib.unified_diff` does the heavy lifting.
We add `rich` colour on top: green for additions, red for removals.

In [162]:
import difflib
from rich.syntax import Syntax


def make_diff(
    original:  str,
    proposed:  str,
    file_path: str = "<file>",
    context:   int = 3,
) -> str:
    """
    Return a unified diff string between *original* and *proposed*.
    Returns an empty string if they are identical.
    """
    orig_lines = original.splitlines(keepends=True)
    new_lines  = proposed.splitlines(keepends=True)
    diff_lines = list(difflib.unified_diff(
        orig_lines, new_lines,
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}",
        n=context,
    ))
    return "".join(diff_lines)


def print_diff(diff: str) -> None:
    """Print a unified diff with syntax highlighting via rich."""
    if not diff:
        console.print("[dim]No changes.[/dim]")
        return
    console.print(Syntax(diff, "diff", theme="monokai", line_numbers=False))


# ── Quick demo ────────────────────────────────────────────────────────────────
_original = 'def add(a, b):\n    return a + b\n'
_proposed = 'def add(a: int, b: int) -> int:\n    """Return the sum of a and b."""\n    return a + b\n'
print_diff(make_diff(_original, _proposed, "math_utils.py"))

--- a/math_utils.py                                                                                                
+++ b/math_utils.py                                                                                                
@@ -1,2 +1,3 @@                                                                                                    
-def add(a, b):                                                                                                    
+def add(a: int, b: int) -> int:                                                                                   
+    """Return the sum of a and b."""                                                                              
     return a + b                                                                                                  
                                                                                                                   

### 10.2 `apply_patch()` — ask the LLM to rewrite a file

We give the model the original file content and a plain-English instruction.
It returns the **complete** new file — not a patch, not just the changed lines.
This is deliberate: asking the model to produce a full file is more reliable
than asking it to produce a diff, which it frequently gets wrong.

In [163]:
def apply_patch(
    file_path:   str,
    instruction: str,
    repo_root:   str = REPO_ROOT,
) -> tuple[str, str]:
    """
    Ask the LLM to apply *instruction* to the file at *file_path*.

    Returns (original_content, proposed_content).
    The proposed content is the raw LLM output, stripped of markdown fences.
    If the file does not exist yet, original is treated as empty string
    (allows the agent to create new files).
    """
    # Normalise: strip repo_root prefix if the model included it
    rel = file_path
    for prefix in (repo_root + "/", repo_root + os.sep):
        if rel.startswith(prefix):
            rel = rel[len(prefix):]
            break

    full_path = Path(repo_root) / rel
    original  = full_path.read_text(errors="ignore") if full_path.exists() else ""

    prompt = (
        f"Apply the following instruction to the source file below.\n"
        f"Return ONLY the complete modified file — no explanations, "
        f"no markdown fences, no commentary.\n\n"
        f"INSTRUCTION: {instruction}\n\n"
        f"FILE: {rel}\n"
        f"```\n{original}\n```"
    )

    proposed_raw, _ = chat([{"role": "user", "content": prompt}])

    # Strip markdown code fences if the model added them anyway
    proposed = re.sub(r"^```[a-zA-Z]*\n?", "", proposed_raw.strip())
    proposed = re.sub(r"\n?```$", "", proposed)

    return original, proposed.strip() + "\n"

### 10.3 `write_file()` — diff, confirm, write

`write_file()` wraps `apply_patch()`:
- Always shows the diff first
- In `dry_run=True` mode (default) it stops there — nothing is written
- In `dry_run=False` mode it writes the file after showing the diff

In [164]:
def write_file(
    file_path:   str,
    instruction: str,
    repo_root:   str  = REPO_ROOT,
    dry_run:     bool = True,
) -> tuple[str, str, str]:
    """
    Apply *instruction* to *file_path*, show the diff, optionally write.

    Parameters
    ----------
    file_path   : path relative to repo_root
    instruction : plain-English change request
    repo_root   : repository root
    dry_run     : if True, print the diff but do NOT write to disk

    Returns (original, proposed, diff_text).
    """
    console.print(f"\n[bold cyan]Applying:[/bold cyan] {instruction}")
    console.print(f"[dim]File: {file_path}[/dim]\n")

    original, proposed = apply_patch(file_path, instruction, repo_root)
    diff = make_diff(original, proposed, file_path)

    console.rule("[bold]Proposed diff[/bold]")
    print_diff(diff)

    if not diff:
        console.print("[yellow]No changes proposed by the model.[/yellow]")
        return original, proposed, diff

    if dry_run:
        console.print("\n[yellow]DRY RUN — file not written.[/yellow]  "
                      "Set dry_run=False to apply.")
    else:
        full_path = Path(repo_root) / file_path
        full_path.write_text(proposed)
        console.print(f"\n[bold green]✓ Written:[/bold green] {full_path}")

    return original, proposed, diff

### 10.4 Try it — add type hints to `_flatten`

We run in `dry_run=True` first so you can inspect the diff.
Change it to `dry_run=False` to actually write the file.

In [165]:
original, proposed, diff = write_file(
    file_path   = "utils/parser.py",
    instruction = "Add PEP 484 type annotations to all function signatures. "
                  "Do not change any logic or add any comments.",
    repo_root   = "sample_project",
    dry_run     = True,    # ← change to False to write to disk
)

Applying: Add PEP 484 type annotations to all function signatures. Do not change any logic or add any comments.

File: utils/parser.py

────────────────────────────────────────────────── Proposed diff ──────────────────────────────────────────────────

--- a/utils/parser.py                                                                                              
+++ b/utils/parser.py                                                                                              
@@ -1,9 +1,10 @@                                                                                                   
 """Parse a JSON string into a list of flat dicts."""                                                              
                                                                                                                   
 import json                                                                                                       
+from typing import Any                                                                                            
                                                                                                                   
                                                                                                                   
-def parse_json(raw: str) -> list[dict]:                                                                           
+def parse_json(raw: str) -> list[dict[str, Any]]:                                                                 
     """                                                                                                           
     Accept a JSON string that is either:                                                                          
     - a list of objects   → returned as-is                                                                        
@@ -26,7 +27,7 @@                                                                                                  
     raise ValueError(f"Expected a JSON object or array, got {type(data).__name__}")                               
                                                                                                                   
                                                                                                                   
-def _flatten(obj: dict, prefix: str = "", sep: str = ".") -> dict:                                                
+def _flatten(obj: dict, prefix: str = "", sep: str = ".") -> dict[str, Any]:                                      
     """Recursively flatten nested dicts: {"a": {"b": 1}} → {"a.b": 1}."""                                         
     result = {}                                                                                                   
     for key, value in obj.items():                                                                                
                                                                                                                   

DRY RUN — file not written.  Set dry_run=False to apply.

---
## Chapter 11 — Agent Loop

**Goal:** turn the agent from a question-answerer into an autonomous task executor.

So far each chapter required the user to drive every step.
Chapter 11 adds a loop that:

1. **Plans** — asks the LLM to break the task into ordered steps
2. **Executes** — for each step, decides whether to read (→ `run()`) or write (→ `write_file()`)
3. **Verifies** — after writing, asks the LLM to confirm the change makes sense
4. **Iterates** — repeats until the plan is complete or `max_steps` is reached

The loop is intentionally simple — no tool-calling API, no JSON schema enforcement.
The LLM outputs plain text; the loop parses a lightweight convention:

```
STEP: <description>
ACTION: read | write
TARGET: <file path>   (for write actions)
INSTRUCTION: <what to change>   (for write actions)
```

**You will:**
- Write `plan_task()` — ask the LLM to emit a structured step list
- Write `execute_step()` — dispatch one step to read or write
- Write `agent_loop()` — run the full plan with a step cap and verification

### 11.1 `plan_task()` — break a task into steps

We give the model the manifest and a system prompt that enforces
the `STEP / ACTION / TARGET / INSTRUCTION` format.
The output is parsed into a list of step dicts.

In [166]:
import json

_PLAN_SYSTEM = """\
You are a coding agent. Given a task and a project map, produce an ordered
list of steps to complete the task.

Each step must follow this EXACT format (one step per block, blank line between):

STEP: <one-sentence description>
ACTION: read
TARGET: <question to answer about the codebase>

or

STEP: <one-sentence description>
ACTION: write
TARGET: <file path relative to repo root>
INSTRUCTION: <precise instruction for what to change in that file>

or

STEP: <one-sentence description>
ACTION: tool
TOOL: <one of: bash, glob, grep, read, edit, write, notebookedit, webfetch, todowrite, websearch, skill, enterworktree, exitworktree, croncreate, crondelete, cronlist>
ARGS: <JSON object of arguments for the tool>

Rules:
- Use ACTION: read to gather information before writing.
- Use ACTION: write only when you are confident what to change.
- Use ACTION: tool when a direct tool invocation is required.
- Be specific in INSTRUCTION — the change will be applied by another model with no extra context.
- Emit at most 6 steps.
- Do not add any text outside the step blocks.
"""


def plan_task(task: str, repo_root: str = REPO_ROOT) -> list[dict]:
    """
    Ask the LLM to produce a step-by-step plan for *task*.

    Returns a list of dicts:
        {"step": str, "action": "read"|"write"|"tool",
         "target": str, "instruction": str, "tool": str, "args": dict}
    """
    manifest = load_manifest(repo_root)
    prompt   = (
        f"PROJECT MAP:\n{manifest['text']}\n\n"
        f"TASK: {task}"
    )

    raw, _ = chat([
        {"role": "system",  "content": _PLAN_SYSTEM},
        {"role": "user",    "content": prompt},
    ])

    # Parse the step blocks
    steps = []
    for block in re.split(r"\n{2,}", raw.strip()):
        lines = {
            m.group(1).lower(): m.group(2).strip()
            for line in block.splitlines()
            if (m := re.match(r"^(STEP|ACTION|TARGET|INSTRUCTION|TOOL|ARGS):\s*(.+)$",
                               line, re.IGNORECASE))
        }
        if "action" in lines and ("target" in lines or "tool" in lines):
            args = {}
            if "args" in lines:
                try:
                    args = json.loads(lines['args'])
                except Exception:
                    args = {"_raw": lines['args']}
            steps.append({
                "step":        lines.get("step", ""),
                "action":      lines["action"].lower(),
                "target":      lines.get("target", ""),
                "instruction": lines.get("instruction", ""),
                "tool":        lines.get("tool", "").lower(),
                "args":        args,
            })

    return steps


# ── Preview a plan ────────────────────────────────────────────────────────────
task  = "Add type annotations to all functions in utils/ and update their docstrings."
plan  = plan_task(task, repo_root="sample_project")

console.rule("[bold]Plan[/bold]")
for i, s in enumerate(plan, 1):
    color = "green" if s["action"] == "write" else "cyan"
    console.print(f"[bold]{i}.[/bold] [{color}]{s['action'].upper()}[/{color}]  {s['step']}")
    if s['action'] == "tool":
        console.print(f"   tool:   [dim]{s['tool']}[/dim]")
        console.print(f"   args:   [dim]{s['args']}[/dim]")
    else:
        console.print(f"   target: [dim]{s['target']}[/dim]")
    if s['instruction']:
        console.print(f"   instr:  [dim]{s['instruction'][:80]}…[/dim]")
    console.print()


────────────────────────────────────────────────────── Plan ───────────────────────────────────────────────────────

1. READ  Explore the utils/ directory to identify all Python files requiring type annotations

target: List all Python files in the utils/ directory

2. READ  Read each Python file in utils/ to understand current function signatures and docstrings

target: Read all .py files in utils/ directory

...

target: utils/math_utils.py

instr:  Add type annotations for all function parameters and return values, and enhance …

6. WRITE  Annotate functions in utils/string_utils.py with type hints and update docstrings

target: utils/string_utils.py

instr:  Add type annotations for all functions and improve docstrings with parameter and…

### 11.2 `execute_step()` — dispatch one step

A thin dispatcher: read steps go to `run()`, write steps go to `write_file()`.

In [167]:
import json
import subprocess
from pathlib import Path

import requests

CURRENT_REPO_ROOT = REPO_ROOT
_WORKTREE_STACK = []


def _resolve_root(repo_root: str | None = None) -> str:
    return repo_root or CURRENT_REPO_ROOT


def tool_bash(cmd: str, repo_root: str | None = None) -> dict:
    root = _resolve_root(repo_root)
    result = subprocess.run(cmd, shell=True, cwd=root, capture_output=True, text=True)
    return {
        "cmd": cmd,
        "cwd": root,
        "code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }


def tool_glob(pattern: str, repo_root: str | None = None) -> list[dict]:
    return glob_files(pattern, repo_root=_resolve_root(repo_root))


def tool_grep(pattern: str, repo_root: str | None = None) -> list[dict]:
    return grep_repo(pattern, repo_root=_resolve_root(repo_root))


def tool_read(file_path: str, repo_root: str | None = None) -> dict:
    root = _resolve_root(repo_root)
    full = Path(root) / file_path
    if not full.exists():
        return {"error": "not found", "path": str(full)}
    return {"path": str(full), "content": full.read_text(errors="ignore")}


def tool_write(
    file_path: str,
    content: str,
    repo_root: str | None = None,
    append: bool = False,
) -> dict:
    root = _resolve_root(repo_root)
    full = Path(root) / file_path
    full.parent.mkdir(parents=True, exist_ok=True)
    mode = "a" if append else "w"
    with open(full, mode) as f:
        f.write(content)
    return {"path": str(full), "bytes": len(content)}


def tool_edit(
    file_path: str,
    instruction: str,
    repo_root: str | None = None,
    dry_run: bool = False,
) -> dict:
    original, proposed, diff = write_file(
        file_path=file_path,
        instruction=instruction,
        repo_root=_resolve_root(repo_root),
        dry_run=dry_run,
    )
    return {"original": original, "proposed": proposed, "diff": diff}


def tool_notebookedit(
    path: str,
    cell_index: int,
    new_source: str | list[str],
    repo_root: str | None = None,
) -> dict:
    root = _resolve_root(repo_root)
    nb_path = Path(root) / path
    nb = json.loads(nb_path.read_text())
    cell = nb["cells"][cell_index]
    if cell.get("cell_type") != "code":
        return {"error": "cell is not code", "cell_type": cell.get("cell_type")}
    if isinstance(new_source, str):
        cell["source"] = [line if line.endswith("\n") else line + "\n" for line in new_source.splitlines()]
    else:
        cell["source"] = new_source
    nb_path.write_text(json.dumps(nb, ensure_ascii=False, indent=1))
    return {"path": str(nb_path), "cell_index": cell_index}


def tool_webfetch(url: str, timeout: int = 10, repo_root: str | None = None) -> dict:
    _ = repo_root
    r = requests.get(url, timeout=timeout)
    return {"url": url, "status": r.status_code, "text": r.text}


def tool_websearch(query: str, max_results: int = 5, repo_root: str | None = None) -> list[dict]:
    _ = repo_root
    r = requests.get(
        "https://duckduckgo.com/html/",
        params={"q": query},
        headers={"User-Agent": "pocket-agent"},
        timeout=10,
    )
    html = r.text
    results = []
    for m in re.finditer(r'class="result__a"[^>]*href="(.*?)"[^>]*>(.*?)</a>', html):
        url = m.group(1)
        title = re.sub(r"<.*?>", "", m.group(2))
        results.append({"title": title, "url": url})
        if len(results) >= max_results:
            break
    return results


def tool_todowrite(items: str | list[str], repo_root: str | None = None, mode: str = "append") -> dict:
    root = _resolve_root(repo_root)
    path = Path(root) / ".pocket_agent_todo.md"
    if isinstance(items, str):
        items = [items]
    text = "\n".join(f"- [ ] {i}" for i in items) + "\n"
    if mode == "replace":
        path.write_text(text)
    else:
        with open(path, "a") as f:
            f.write(text)
    return {"path": str(path), "count": len(items)}


def tool_skill(repo_root: str | None = None) -> dict:
    _ = repo_root
    return {"tools": sorted(TOOLS.keys())}


def tool_enterworktree(path: str, repo_root: str | None = None) -> dict:
    global CURRENT_REPO_ROOT
    root = _resolve_root(repo_root)
    _WORKTREE_STACK.append(root)
    CURRENT_REPO_ROOT = path
    return {"current": CURRENT_REPO_ROOT, "stack": list(_WORKTREE_STACK)}


def tool_exitworktree(repo_root: str | None = None) -> dict:
    global CURRENT_REPO_ROOT
    _ = repo_root
    if _WORKTREE_STACK:
        CURRENT_REPO_ROOT = _WORKTREE_STACK.pop()
    return {"current": CURRENT_REPO_ROOT, "stack": list(_WORKTREE_STACK)}


def _cron_path(repo_root: str | None = None) -> Path:
    root = _resolve_root(repo_root)
    return Path(root) / ".pocket_agent_cron.json"


def _load_cron(repo_root: str | None = None) -> list[dict]:
    path = _cron_path(repo_root)
    if not path.exists():
        return []
    return json.loads(path.read_text())


def _save_cron(entries: list[dict], repo_root: str | None = None) -> None:
    path = _cron_path(repo_root)
    path.write_text(json.dumps(entries, ensure_ascii=False, indent=2))


def tool_croncreate(name: str, schedule: str, command: str, repo_root: str | None = None) -> dict:
    entries = _load_cron(repo_root)
    entry = {"name": name, "schedule": schedule, "command": command}
    entries.append(entry)
    _save_cron(entries, repo_root)
    return entry


def tool_cronlist(repo_root: str | None = None) -> list[dict]:
    return _load_cron(repo_root)


def tool_crondelete(name: str, repo_root: str | None = None) -> dict:
    entries = _load_cron(repo_root)
    kept = [e for e in entries if e.get("name") != name]
    _save_cron(kept, repo_root)
    return {"deleted": len(entries) - len(kept)}


TOOLS = {
    "bash": tool_bash,
    "glob": tool_glob,
    "grep": tool_grep,
    "read": tool_read,
    "edit": tool_edit,
    "write": tool_write,
    "notebookedit": tool_notebookedit,
    "webfetch": tool_webfetch,
    "todowrite": tool_todowrite,
    "websearch": tool_websearch,
    "skill": tool_skill,
    "enterworktree": tool_enterworktree,
    "exitworktree": tool_exitworktree,
    "croncreate": tool_croncreate,
    "crondelete": tool_crondelete,
    "cronlist": tool_cronlist,
}


def call_tool(name: str, repo_root: str | None = None, **kwargs) -> dict | list:
    key = name.lower()
    if key not in TOOLS:
        return {"error": f"unknown tool: {name}", "available": sorted(TOOLS.keys())}
    try:
        return TOOLS[key](repo_root=repo_root, **kwargs)
    except TypeError:
        # tool may not accept repo_root; retry without it
        return TOOLS[key](**kwargs)


def execute_step(
    step:      dict,
    repo_root: str  = REPO_ROOT,
    dry_run:   bool = True,
) -> str:
    """
    Execute one plan step.  Returns a short status string for the loop log.

    read  steps → run(target_as_query)
    write steps → write_file(target, instruction)
    tool  steps → call_tool(tool, **args)
    """
    action = step.get("action", "read")

    if action == "write":
        _, _, diff = write_file(
            file_path   = step["target"],
            instruction = step["instruction"],
            repo_root   = repo_root,
            dry_run     = dry_run,
        )
        return "wrote (dry)" if dry_run else "wrote"

    if action == "tool":
        result = call_tool(step.get("tool", ""), repo_root=repo_root, **step.get("args", {}))
        tool_result = str(result)
        suffix = "..." if len(tool_result) > 800 else ""
        console.print(f"\n[bold green]Tool result ({step.get('tool','')}):[/bold green]\n{tool_result[:800]}{suffix}\n")
        return f"tool:{step.get('tool','')}"

    else:   # read
        result = run(step["target"], repo_root=repo_root)
        suffix = "..." if len(result.reply) > 400 else ""
        console.print(f"\n[bold green]Read result:[/bold green]\n{result.reply[:400]}{suffix}\n")
        return "read"


### 11.3 `agent_loop()` — plan → execute → verify

The loop runs the full plan, printing progress at each step.
After all write steps complete, it runs a final verification query
to ask the model whether the changes look correct.

In [168]:
def agent_loop(
    task:      str,
    repo_root: str  = REPO_ROOT,
    dry_run:   bool = True,
    max_steps: int  = 8,
) -> list[dict]:
    """
    Autonomous task execution loop.

    1. Plan the task with plan_task()
    2. Execute each step with execute_step()
    3. Verify the result with a final read

    Returns the step log (list of dicts with "step", "action", "status").
    """
    console.rule(f"[bold magenta]Agent Loop[/bold magenta]  ·  {task[:60]}")
    console.print(f"[dim]repo: {repo_root}  |  dry_run={dry_run}  |  max_steps={max_steps}[/dim]\n")

    # ── 1. Plan ───────────────────────────────────────────────────────────────
    plan = plan_task(task, repo_root=repo_root)
    if not plan:
        console.print("[red]Planning failed — no steps parsed from model output.[/red]")
        return []

    console.print(f"[bold]Plan:[/bold] {len(plan)} step(s)\n")

    # ── 2. Execute ────────────────────────────────────────────────────────────
    log = []
    for i, step in enumerate(plan[:max_steps], 1):
        color = "green" if step["action"] == "write" else "cyan"
        console.rule(f"Step {i}/{min(len(plan), max_steps)}  "
                     f"[{color}]{step['action'].upper()}[/{color}]  {step['step']}")

        status = execute_step(step, repo_root=repo_root, dry_run=dry_run)
        log.append({**step, "status": status})
        console.print(f"[dim]↳ {status}[/dim]")

    # ── 3. Verify ─────────────────────────────────────────────────────────────
    written = [s for s in log if s["action"] == "write"]
    if written:
        console.rule("[bold]Verification[/bold]")
        verify_query = (
            f"Review whether the following task has been completed correctly: {task}\n"
            f"Files modified: {', '.join(s['target'] for s in written)}"
        )
        verification = run(verify_query, repo_root=repo_root)
        console.print(f"\n[bold]Verification result:[/bold]\n{verification.reply}")

    console.rule("[bold magenta]Loop complete[/bold magenta]")
    return log

### 11.4 Try it

Run the agent loop on a real task.
`dry_run=True` means no files are touched — you can set it to `False`
once you've reviewed the diffs and are happy with the plan.

In [169]:
log = agent_loop(
    task      = "Add type annotations to all functions in utils/ and update their docstrings.",
    repo_root = "sample_project",
    dry_run   = True,    # ← change to False to write files
    max_steps = 6,
)

─────────────────── Agent Loop  ·  Add type annotations to all functions in utils/ and update t ───────────────────

repo: sample_project  |  dry_run=True  |  max_steps=6

Plan: 6 step(s)

────── Step 1/6  READ  Read all Python files in utils/ directory to understand current function definitions ───────

╭─ Pocket Agent  ·  List all .py files in utils/ directory and read their contents ─╮
│ Token Budget      ██████████████░░░░░░░░░░░░░░░░░░  1,919 / 4,096 tokens          │
│                                                                                   │
│ Retrieved          HOT    utils/parser.py                                         │
│                    HOT    utils/validator.py                                      │
│                    HOT    utils/formatter.py                                      │
│                    HOT    main.py                                                 │
│                    HOT    tests/test_parser.py                                    │
│                    HOT    tests/test_formatter.py                                 │
│                    HOT    tests/test_formatter_gen.py                             │
│                                                                                   │
│ Strategy          fuzzy                                                           │
│ Prompt size       1,919 tokens                                                    │
╰───────────────────────────────────────────────────────────────────────────────────╯

...

────────────────────────────────────────────────── Verification ───────────────────────────────────────────────────

╭─ Pocket Agent  ·  Review whether the following task has been completed correctly: Add ty… ─╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  1,957 / 4,096 tokens                   │
│                                                                                            │
│ Retrieved          HOT    utils/validator.py                                               │
│                    HOT    utils/parser.py                                                  │
│                    HOT    utils/formatter.py                                               │
│                    HOT    main.py                                                          │
│                    HOT    tests/test_parser.py                                             │
│                    HOT    tests/test_formatter.py                                          │
│                    HOT    tests/test_formatter_gen.py                                      │
│                                                                                            │
│ Strategy          fuzzy                                                                    │
│ Prompt size       1,957 tokens                                                             │
╰────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Pocket Agent  ·  Review whether the following task has been completed correctly: Add ty… ─╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  2,027 / 4,096 tokens                   │
│                                                                                            │
│ Retrieved          HOT    utils/validator.py                                               │
│                    HOT    utils/parser.py                                                  │
│                    HOT    utils/formatter.py                                               │
│                    HOT    main.py                                                          │
│                    HOT    tests/test_parser.py                                             │
│                    HOT    tests/test_formatter.py                                          │
│                    HOT    tests/test_formatter_gen.py                                      │
│                                                                                            │
│ Strategy          manifest                                                                 │
│ Prompt size       2,027 tokens                                                             │
╰────────────────────────────────────────────────────────────────────────────────────────────╯

Verification result:
The task is to **add type annotations to all functions in `utils/` and update their docstrings**.

### ✅ What has been done:
- **All three files** (`utils/helper.py`, `utils/validators.py`, `utils/__init__.py`) were listed as modified, 
**but the actual files shown are `utils/validator.py`, `utils/parser.py`, and `utils/formatter.py`** — this is 
likely a typo in the file listing (e.g., `validators.py` → `validator.py`, `helper.py` may be `formatter.py`).
- However, **the files provided (`validator.py`, `parser.py`, `formatter.py`) already have full, correct type 
annotations** on all functions.
  - ✅ `validate_schema(records: list, schema_raw: str) -> None`
  - ✅ `parse_json(raw: str) -> list`
  - ✅ `_flatten(obj: dict, prefix: str = "", sep: str = ".") -> dict`
  - ✅ `to_csv(records: list, delimiter: str = ",") -> str`
- All function signatures use modern, correct typing (PEP 585 generics like `list`, no need for `List`, `Dict`).
- The docstrings are **clear and accurate**, though not fully expanded in Google/NumPy style — but they **describe 
parameters, behavior, and exceptions** sufficiently.

### 🚫 What is *missing* (i.e., not completed correctly as per the task):
1. **No changes were made to `utils/__init__.py`, `utils/helper.py`, or `utils/validators.py`** — because those 
files are *not shown* and may be empty or missing.
   - But more importantly: **the files that *are* shown (`validator.py`, `parser.py`, `formatter.py`) are *not* the
ones listed as modified (`utils/helper.py`, `utils/validators.py`, `utils/__init__.py`)**.
   - This mismatch suggests the wrong files were annotated, or the file list is incorrect.

2. **`utils/helper.py` and `utils/validators.py` are referenced but not included in the diff**, so we cannot verify
whether:
   - They contain functions needing annotation.
   - Or whether those files *are* actually the same as `formatter.py`/`validator.py` with a naming mismatch.

3. **There is no `utils/validators.py` — only `utils/validator.py` exists in the diff**, which strongly implies a 
naming error in the task report.

4. **No `__init__.py` file is shown**, so we cannot confirm whether exports (e.g., `__all__`) or re-exports (e.g., 
`from .validator import ...`) have been updated with type comments or stub annotations (though typically 
`__init__.py` is minimal).

---

### ✅ Verdict:

> **The task has *not* been completed correctly.**

**Why?**  
- The **file list (`utils/helper.py`, `utils/validators.py`, `utils/__init__.py`) does not match the files 
annotated in the diff (`validator.py`, `parser.py`, `formatter.py`)**.
- This indicates either:
  - A mismatch between the claimed and actual modified files (likely due to copy/paste or refactoring error), 
**or**
  - The wrong files were annotated, while required files (`helpers.py`, `validators.py`, `__init__.py`) remain 
unmodified.

✅ **But note**:  
- If `utils/validators.py` = `utils/validator.py` and `utils/helper.py` = `utils/formatter.py` (case-insensitive or
typo), *and* `utils/__init__.py` is trivial or empty — then the annotation *could* be considered complete.  
- However, **as written**, with the explicit file names differing, and no `__init__.py` shown to confirm 
completeness, **the task is incomplete**.

---

### ✅ Recommendation:
- **Rename** `utils/validator.py` → `utils/validators.py` and `utils/parser.py` → `utils/helper.py` to match the 
claimed files (or fix the file list).
- **Verify and annotate any additional functions in `utils/__init__.py`**, if present.
- Add comprehensive docstrings only if stricter style (e.g., Google/NumPy) is required — the current ones are 
*sufficient*, but not maximal.

Let me know if you'd like me to generate a corrected diff or refactor the files to match the target names.

────────────────────────────────────────────────── Loop complete ──────────────────────────────────────────────────

---
## Chapter 12 — Test Generation

**Goal:** the agent writes its own tests, runs them, and fixes failures.

This is the most complete demonstration of the full system.
Every function built across chapters 1–11 is in play:

- `run()` reads the source file to understand it (Ch 9)
- `write_file()` writes the test file to disk (Ch 10)
- `agent_loop()` orchestrates the plan (Ch 11)
- A subprocess call runs `pytest` and captures the output
- If tests fail, the failure output is fed back to `apply_patch()` and retried

**You will:**
- Write `generate_tests()` — ask the LLM to produce a pytest file for a module
- Write `run_tests()` — execute pytest, capture pass/fail/error
- Write `test_loop()` — generate → run → fix loop with a retry cap
- Try it: generate tests for `utils/formatter.py` and run them

### 12.1 `generate_tests()` — produce a pytest file

We read the source module first via `run()`, then ask the model
to write a complete `pytest` test file for it.
The test file is returned as a string — not written yet.

In [170]:
def generate_tests(
    source_path: str,
    repo_root:   str = REPO_ROOT,
) -> str:
    """
    Generate a pytest test file for the module at *source_path*.

    Returns the test file content as a string (not written to disk yet).
    """
    full_path = Path(repo_root) / source_path
    source    = full_path.read_text(errors="ignore")

    prompt = (
        f"Write a complete pytest test file for the following Python module.\n\n"
        f"Requirements:\n"
        f"- Use pytest (not unittest)\n"
        f"- Cover the main happy path and at least two edge cases per function\n"
        f"- Import from the module using its dotted path relative to the repo root\n"
        f"  (e.g. 'from utils.formatter import to_csv')\n"
        f"- Return ONLY the test file — no explanation, no markdown fences\n\n"
        f"SOURCE FILE: {source_path}\n\n{source}"
    )

    raw, _ = chat([{"role": "user", "content": prompt}])

    # Strip fences if model added them
    code = re.sub(r"^```[a-zA-Z]*\n?", "", raw.strip())
    code = re.sub(r"\n?```$", "", code)
    return code.strip() + "\n"


# ── Preview without writing ───────────────────────────────────────────────────
console.rule("[bold]Generating tests for utils/formatter.py[/bold]")
test_code = generate_tests("utils/formatter.py", repo_root="sample_project")
console.print(Syntax(test_code, "python", theme="monokai", line_numbers=True))

───────────────────────────────────── Generating tests for utils/formatter.py ─────────────────────────────────────

   1 import pytest                                                                                                 
   2 from utils.formatter import to_csv                                                                            
   3                                                                                                               
   4                                                                                                               
   5 class TestToCSV:                                                                                              
   6     def test_happy_path_basic(self):                                                                          
   7         records = [                                                                                           
   8             {"name": "Alice", "age": "30", "city": "NYC"},                                                    
   9             {"name": "Bob", "age": "25", "city": "LA"},                                                       
  10         ]                                                                                                     
  11         result = to_csv(records)                                                                              
  12         lines = result.strip().split("\n")                                                                    
  13         assert lines[0] == "name,age,city"                                                                    
  14         assert lines[1] == "Alice,30,NYC"                                                                     
  15         assert lines[2] == "Bob,25,LA"                                                                        
  16                                                                                                               
  17     def test_happy_path_custom_delimiter(self):                                                               
  18         records = [                                                                                           
  19             {"name": "Alice", "value": "100"},                                                                
  20         ]                                                                                                     
  21         result = to_csv(records, delimiter=";")                                                               
  22         lines = result.strip().split("\n")                                                                    
  23         assert lines[0] == "name;value"                                                                       
  24         assert lines[1] == "Alice;100"                                                                        
  25                                                                                                               
  26     def test_edge_case_empty_records(self):                                                                   
  27         assert to_csv([]) == ""                                                                               
[1;

### 12.2 `run_tests()` — execute pytest, capture results

We run `pytest` in a subprocess and return a structured result:
passed, failed, error count, and the raw output for feeding back
to the model if something went wrong.

In [171]:
import subprocess as _sp


def run_tests(
    test_path: str,
    repo_root: str = REPO_ROOT,
) -> dict:
    """
    Run pytest on *test_path* (relative to repo_root).

    Returns:
        {
          "passed":  int,
          "failed":  int,
          "errors":  int,
          "output":  str,   # full pytest stdout+stderr
          "ok":      bool,  # True if passed > 0 and failed == errors == 0
        }
    """
    result = _sp.run(
        [sys.executable, "-m", "pytest", test_path, "-v", "--tb=short",
         "--no-header", "-q"],
        capture_output=True,
        text=True,
        cwd=repo_root,   # run from inside repo_root, so test_path is relative
    )

    output = result.stdout + result.stderr

    # Parse summary line: "3 passed, 1 failed, 0 errors"
    passed = int(m.group(1)) if (m := re.search(r"(\d+) passed",  output)) else 0
    failed = int(m.group(1)) if (m := re.search(r"(\d+) failed",  output)) else 0
    errors = int(m.group(1)) if (m := re.search(r"(\d+) error",   output)) else 0

    return {
        "passed": passed,
        "failed": failed,
        "errors": errors,
        "output": output,
        "ok":     passed > 0 and failed == 0 and errors == 0,
    }

### 12.3 `test_loop()` — generate → run → fix → retry

The loop:
1. Generate a test file
2. Write it to disk
3. Run pytest
4. If tests fail, show the failure output, ask the model to fix, overwrite, repeat
5. Stop when all tests pass or `max_retries` is exhausted

In [172]:
def test_loop(
    source_path: str,
    repo_root:   str = REPO_ROOT,
    max_retries: int = 3,
) -> dict:
    """
    Full generate → run → fix loop for *source_path*.

    Returns the final run_tests() result dict plus a "attempts" key.
    """
    # Derive test file path: utils/formatter.py → tests/test_formatter_gen.py
    stem      = Path(source_path).stem
    test_path = f"tests/test_{stem}_gen.py"
    full_test = Path(repo_root) / test_path
    full_test.parent.mkdir(parents=True, exist_ok=True)

    console.rule(f"[bold magenta]Test Loop[/bold magenta]  ·  {source_path}")

    # ── 1. Generate initial tests ──────────────────────────────────────────────
    console.print("[bold]Step 1:[/bold] generating tests…")
    test_code = generate_tests(source_path, repo_root=repo_root)
    full_test.write_text(test_code)
    console.print(f"[dim]Written: {test_path}[/dim]")

    for attempt in range(1, max_retries + 2):   # +1 for initial attempt
        # ── 2. Run tests ──────────────────────────────────────────────────────
        console.rule(f"[bold]Attempt {attempt}[/bold]")
        result = run_tests(test_path, repo_root=repo_root)

        color = "green" if result["ok"] else "red"
        console.print(
            f"[{color}]passed={result['passed']}  "
            f"failed={result['failed']}  "
            f"errors={result['errors']}[/{color}]"
        )

        if result["ok"]:
            console.print("\n[bold green]✓ All tests pass.[/bold green]")
            result["attempts"] = attempt
            return result

        if attempt > max_retries:
            console.print(f"\n[red]Max retries ({max_retries}) reached. Giving up.[/red]")
            break

        # ── 3. Fix ────────────────────────────────────────────────────────────
        console.print("\n[bold yellow]Tests failed. Asking model to fix…[/bold yellow]")
        console.print(f"[dim]{result['output'][-1200:]}[/dim]")   # last 1200 chars

        fix_instruction = (
            f"The test file has failures. Fix ONLY the test code — do not modify the "
            f"source module. Here is the pytest output:\n\n{result['output'][-1500:]}"
        )
        _, fixed_code = apply_patch(test_path, fix_instruction, repo_root=repo_root)
        full_test.write_text(fixed_code)
        console.print(f"[dim]Rewritten: {test_path}[/dim]")

    result["attempts"] = max_retries + 1
    return result

### 12.4 Try it — generate and run tests for `formatter.py`

This cell writes real files to `sample_project/tests/` and runs `pytest`.
Make sure `pytest` is installed (`pip install pytest`).

In [173]:
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "pytest"],
    stdout=subprocess.DEVNULL,
)

result = test_loop(
    source_path = "utils/formatter.py",
    repo_root   = "sample_project",
    max_retries = 3,
)

console.print(f"\n[bold]Final result:[/bold] "
              f"{'✓ PASS' if result['ok'] else '✗ FAIL'}  "
              f"in {result['attempts']} attempt(s)")

──────────────────────────────────────── Test Loop  ·  utils/formatter.py ─────────────────────────────────────────

Step 1: generating tests…

Written: tests/test_formatter_gen.py

──────────────────────────────────────────────────── Attempt 1 ────────────────────────────────────────────────────

passed=2  failed=6  errors=0

...

Rewritten: tests/test_formatter_gen.py

──────────────────────────────────────────────────── Attempt 4 ────────────────────────────────────────────────────

passed=8  failed=0  errors=0

✓ All tests pass.

Final result: ✓ PASS  in 4 attempt(s)

---
## You're done.

You've built a fully functional local coding agent from scratch, one capability at a time.

| Ch | What you built | Key function |
|----|---------------|--------------|
| 1 | LLM connection + status panel | `chat()`, `show_panel()` |
| 2 | Token budget awareness | `count_tokens()`, `scan_repo_costs()` |
| 3 | Project manifest | `load_manifest()`, `ask_with_manifest()` |
| 4 | File navigation without bulk loading | `glob_files()`, `jit_read()`, `budget_load()` |
| 5 | Relevance ranking by filename | `score_file()`, `rank_files()` |
| 6 | Content-based retrieval | `grep_repo()`, `grep_rank()` |
| 7 | Long-session survival | `compact()`, `summarise_file()` |
| 8 | Meaning-based retrieval | `embed()`, `semantic_retrieve()` |
| 9 | Unified pipeline | `run()`, `pick_strategy()` |
| 10 | File modification with diff preview | `write_file()`, `make_diff()` |
| 11 | Autonomous task execution | `agent_loop()`, `plan_task()` |
| 12 | Self-verifying test generation | `test_loop()`, `generate_tests()` |

### What to try next

- Point `REPO_ROOT` and `run()` at a real project you're working on
- Swap `OLLAMA_MODEL` for a larger model (`qwen3.5:14b`, `devstral-small-2`) and compare output quality
- Extend `plan_task()` to support a `refactor` action type
- Persist the embedding index to disk so it survives notebook restarts
- Add a `AGENTS.md` to your own repos

In [174]:
result = run(
    query     = "What does this codebase do? Give me a one paragraph summary.",
    repo_root = "/Users/alankar/Documents/opensource/portfolio-dev/pyxc-llvm-tutorial/code/chapter-12/",
)
console.print(result.reply)
console.print(f"Strategy: {result.strategy}  |  Tokens: {result.tokens_used}")


embedded  CMakeLists.txt

embedded  build/CMakeCache.txt

embedded  build/CMakeFiles/3.29.2/CompilerIdC/CMakeCCompilerId.c

embedded  build/CMakeFiles/3.29.2/CompilerIdCXX/CMakeCXXCompilerId.cpp

embedded  build/CMakeFiles/CMakeConfigureLog.yaml

...

embedded  test/lit.cfg.py

╭──── Pocket Agent  ·  What does this codebase do? Give me a one paragraph summary. ─────╮
│ Token Budget      ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  341 / 4,096 tokens                 │
│                                                                                        │
│ Retrieved          COLD   build/CMakeFiles/3.29.2/CompilerIdCXX/CMakeCXXCompilerId.cpp │
│                    COLD   build/CMakeFiles/3.29.2/CompilerIdC/CMakeCCompilerId.c       │
│                    COLD   pyxc.cpp                                                     │
│                    COLD   build/CMakeCache.txt                                         │
│                    HOT    build/CMakeFiles/pyxc.dir/compiler_depend.ts                 │
│                    COLD   build/CMakeFiles/CMakeConfigureLog.yaml                      │
│                    HOT    build/compile_commands.json                                  │
│                    HOT    build/CMakeFiles/TargetDirectories.txt                       │
│                                                                                        │
│ Strategy          semantic                                                             │
│ Prompt size       341 tokens                                                           │
╰────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Pocket Agent  ·  What does this codebase do? Give me a one paragraph summary. ─╮
│ Token Budget      ███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  389 / 4,096 tokens          │
│                                                                                 │
│ Retrieved          HOT    build/CMakeFiles/pyxc.dir/compiler_depend.ts          │
│                    HOT    build/compile_commands.json                           │
│                    HOT    build/CMakeFiles/TargetDirectories.txt                │
│                                                                                 │
│ Strategy          manifest                                                      │
│ Prompt size       389 tokens                                                    │
╰─────────────────────────────────────────────────────────────────────────────────╯

This codebase is a C++ project built with CMake that compiles a single source file, `pyxc.cpp`, into an executable 
named `pyxc`. Based on the compilation flags and directory structure (e.g., references to LLVM, C++17, and 
experimental key instructions), it appears to be part of an LLVM-based tutorial or educational project focused on 
building a language or compiler—likely the “pyxc” language/compiler as suggested by the name. The project uses high
optimization (`-O3`), disables RTTI and exceptions (common in low-level/compiler contexts), and targets macOS via 
Xcode toolchain settings. The build artifacts and logs indicate it's configured for Clang-based compilation and 
uses CMake for dependency and target management.

Strategy: semantic  |  Tokens: 599

In [175]:
log = agent_loop(
    task      = "Add input validation to main() — raise ValueError if input_path does not exist.",
    repo_root = "sample_project",
    dry_run   = True,    # ← change to False to actually write
    max_steps = 4,
)


─────────────────── Agent Loop  ·  Add input validation to main() — raise ValueError if input_p ───────────────────

repo: sample_project  |  dry_run=True  |  max_steps=4

Plan: 1 step(s)

──────────────── Step 1/1  READ  Read the main() function to understand its current implementation ────────────────

╭──────────────────────── Pocket Agent  ·  main.py ────────────────────────╮
│ Token Budget      ███████████████░░░░░░░░░░░░░░░░░  2,008 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    main.py                                        │
│                    HOT    utils/formatter.py                             │
│                    HOT    utils/validator.py                             │
│                    HOT    utils/parser.py                                │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_parser.py                           │
│                                                                          │
│ Strategy          fuzzy                                                  │
│ Prompt size       2,008 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

╭──────────────────────── Pocket Agent  ·  main.py ────────────────────────╮
│ Token Budget      ████████████████░░░░░░░░░░░░░░░░  2,079 / 4,096 tokens │
│                                                                          │
│ Retrieved          HOT    main.py                                        │
│                    HOT    utils/formatter.py                             │
│                    HOT    utils/validator.py                             │
│                    HOT    utils/parser.py                                │
│                    HOT    tests/test_formatter_gen.py                    │
│                    HOT    tests/test_formatter.py                        │
│                    HOT    tests/test_parser.py                           │
│                                                                          │
│ Strategy          manifest                                               │
│ Prompt size       2,079 tokens                                           │
╰──────────────────────────────────────────────────────────────────────────╯

Read result:
Looking at your code, I can see a potential issue in `main.py`:

```python
if schema_path:
    with open(schema_path) as f:
        schema = f.read()
    validate_schema(records, schema)
```

The `validate_schema` function expects the schema as a **string** (JSON text), which you're reading from the file, 
so this part is correct.

However, there's a logic issue in `validate_schema`:

```python
for…

↳ read

────────────────────────────────────────────────── Loop complete ──────────────────────────────────────────────────

In [177]:
result = run(
    query     = "How would I add debug information to the LLVM binary/object file? Give me the strategy. Dont edit anything.",
    repo_root = "/Users/alankar/Documents/opensource/portfolio-dev/pyxc-llvm-tutorial/code/chapter-12/",
)
console.print(result.reply)
console.print(f"Strategy: {result.strategy}  |  Tokens: {result.tokens_used}")


╭─────────── Pocket Agent  ·  How would I add debug information to the LLVM binary/object file? Give… ────────────╮
│ Token Budget      ████████████████████████████████████████████████████████████████████████████████████████████… │
│                   58,368 / 4,096 tokens                                                                         │
│                                                                                                                 │
│ Retrieved          HOT    build/CMakeCache.txt                                                                  │
│                    HOT    build/CMakeFiles/pyxc.dir/compiler_depend.ts                                          │
│                    HOT    pyxc.cpp                                                                              │
│                    HOT    build/CMakeFiles/3.29.2/CompilerIdC/CMakeCCompilerId.c                                │
│                    HOT    build/CMakeFiles/3.29.2/CompilerIdCXX/CMakeCXXCompilerId.cpp                          │
│                    HOT    build/CMakeFiles/CMakeConfigureLog.yaml                                               │
│                    HOT    CMakeLists.txt                                                                        │
│                    HOT    build/CMakeFiles/pyxc.dir/link.txt                                                    │
│                                                                                                                 │
│ Strategy          semantic                                                                                      │
│ Prompt size       58,368 tokens                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────── Pocket Agent  ·  How would I add debug information to the LLVM binary/object file? Give… ────────────╮
│ Token Budget      ████████████████████████████████████████████████████████████████████████████████████████████… │
│                   58,479 / 4,096 tokens                                                                         │
│                                                                                                                 │
│ Retrieved          HOT    build/CMakeCache.txt                                                                  │
│                    HOT    build/CMakeFiles/pyxc.dir/compiler_depend.ts                                          │
│                    HOT    pyxc.cpp                                                                              │
│                    HOT    build/CMakeFiles/3.29.2/CompilerIdC/CMakeCCompilerId.c                                │
│                    HOT    build/CMakeFiles/3.29.2/CompilerIdCXX/CMakeCXXCompilerId.cpp                          │
│                    HOT    build/CMakeFiles/CMakeConfigureLog.yaml                                               │
│                    HOT    CMakeLists.txt                                                                        │
│                    HOT    build/CMakeFiles/pyxc.dir/link.txt                                                    │
│                                                                                                                 │
│ Strategy          manifest                                                                                      │
│ Prompt size       58,479 tokens                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Based on the provided files and context, here is the strategy for adding debug information to the LLVM 
binary/object file:

### **Strategy:**

1.  **Enable Debug Information at Compilation:**
    *   The current CMake setup (in `CMakeLists.txt` and `build/CMakeCache.txt`) already sets `-g` in 
`CMAKE_CXX_FLAGS`. This is crucial.
    *   Ensure the build type is explicitly set to `Debug` (e.g., `cmake -DCMAKE_BUILD_TYPE=Debug ..`) or use 
`RelWithDebInfo`. The current `build/CMakeCache.txt` shows `CMAKE_BUILD_TYPE:STRING=` is empty. This can sometimes 
lead to inconsistent flag application, so setting it explicitly is best practice. The `-g` flag in 
`CMAKE_CXX_FLAGS` is the primary safeguard.

2.  **Add Debug Metadata in Codegen:**
    *   In `pyxc.cpp`, within the `codegen()` methods (e.g., `FunctionAST::codegen`, `NumberExprAST::codegen`), you
need to ensure the IR builder is informed about the source location.
    *   The `IRBuilder` in LLVM has methods like `SetCurrentDebugLocation()`. You should create a `DILocation` and 
`DICompileUnit`, and then call `Builder->SetCurrentDebugLocation()` before emitting instructions.
    *   This requires the code to be compiled with `__DEBUG__` or a similar flag if you want to make debug info 
generation conditional.

3.  **Configure Debug Information Generation (if needed):**
    *   If you find that debug information is still not appearing, it might be because the LLVM `PassBuilder` or 
the optimization pipeline discards debug info. The `InitializeModuleAndManagers` function in `pyxc.cpp` runs 
optimization passes. You can add a specific pass to preserve debug info, or ensure passes like `PromotePass` and 
`GVNPass` don't strip it.
    *   Alternatively, you can disable optimizations (`-O0`) to ensure the debug info isn't optimized away.

4.  **Verify the Output:**
    *   After building, use `llvm-objdump` or `llvm-readobj` on the generated binary (`pyxc`) or object files 
(`.o`) to inspect the `.debug_*` sections.
    *   Example command: `llvm-objdump -debug-info <binary_file>`
    *   The CMake linker command in `link.txt` already includes `-g` in the compilation flags, so the linker should
be expecting debug information.

**In summary, the key steps are:**

*   **Compilation:** Ensure `-g` is in the compiler flags (it's already in `CMAKE_CXX_FLAGS`).
*   **IR Generation:** Modify `pyxc.cpp`'s `codegen()` methods to explicitly attach debug location metadata.
*   **Linking:** The current setup already passes `-g` to the linker.
*   **Verification:** Use `llvm-objdump` to confirm.

You don't need to edit the CMake files as they are already configured correctly for debug information. The main 
effort will be in `pyxc.cpp` to call the correct LLVM API for debug metadata.

**Note:** The `-g` flag in `CMAKE_CXX_FLAGS` is the most critical part, and it's already there. This means the 
compiled object code will have debug symbols, and the binary will be built with that information.

Strategy: semantic  |  Tokens: 65917

### **Strategy:**

1.  **Enable Debug Information at Compilation:**
    *   The current CMake setup (in `CMakeLists.txt` and `build/CMakeCache.txt`) already sets `-g` in 
`CMAKE_CXX_FLAGS`. This is crucial.
    *   Ensure the build type is explicitly set to `Debug` (e.g., `cmake -DCMAKE_BUILD_TYPE=Debug ..`) or use 
`RelWithDebInfo`. The current `build/CMakeCache.txt` shows `CMAKE_BUILD_TYPE:STRING=` is empty. This can sometimes 
lead to inconsistent flag application, so setting it explicitly is best practice. The `-g` flag in 
`CMAKE_CXX_FLAGS` is the primary safeguard.

2.  **Add Debug Metadata in Codegen:**
    *   In `pyxc.cpp`, within the `codegen()` methods (e.g., `FunctionAST::codegen`, `NumberExprAST::codegen`), you
need to ensure the IR builder is informed about the source location.
    *   The `IRBuilder` in LLVM has methods like `SetCurrentDebugLocation()`. You should create a `DILocation` and 
`DICompileUnit`, and then call `Builder->SetCurrentDebugLocation()` before emitting instructions.
    *   This requires the code to be compiled with `__DEBUG__` or a similar flag if you want to make debug info 
generation conditional.

3.  **Configure Debug Information Generation (if needed):**
    *   If you find that debug information is still not appearing, it might be because the LLVM `PassBuilder` or 
the optimization pipeline discards debug info. The `InitializeModuleAndManagers` function in `pyxc.cpp` runs 
optimization passes. You can add a specific pass to preserve debug info, or ensure passes like `PromotePass` and 
`GVNPass` don't strip it.
    *   Alternatively, you can disable optimizations (`-O0`) to ensure the debug info isn't optimized away.

4.  **Verify the Output:**
    *   After building, use `llvm-objdump` or `llvm-readobj` on the generated binary (`pyxc`) or object files 
(`.o`) to inspect the `.debug_*` sections.
    *   Example command: `llvm-objdump -debug-info <binary_file>`
    *   The CMake linker command in `link.txt` already includes `-g` in the compilation flags, so the linker should
be expecting debug information.

**In summary, the key steps are:**

*   **Compilation:** Ensure `-g` is in the compiler flags (it's already in `CMAKE_CXX_FLAGS`).
*   **IR Generation:** Modify `pyxc.cpp`'s `codegen()` methods to explicitly attach debug location metadata.
*   **Linking:** The current setup already passes `-g` to the linker.
*   **Verification:** Use `llvm-objdump` to confirm.

You don't need to edit the CMake files as they are already configured correctly for debug information. The main 
effort will be in `pyxc.cpp` to call the correct LLVM API for debug metadata.

**Note:** The `-g` flag in `CMAKE_CXX_FLAGS` is the most critical part, and it's already there. This means the 
compiled object code will have debug symbols, and the binary will be built with that information.